# Notebook 3 — Graph Neural Networks for Solvation Free Energy Prediction

## Aim

This notebook applies two graph neural network architectures — **ALIGNN** and the **Molecular Graph Transformer (MGT)** — to the same solute–solvent solvation free energy task analysed in Notebook 2 using engineered RDKit descriptors and classical/ensemble regressors.

The research question is whether **learned molecular graph representations** can match or exceed the **engineered descriptor pipeline** from Notebook 2.

## Important caveat acknowledged upfront

ALIGNN and MGT were originally designed and benchmarked mainly for **materials and crystal-structure property prediction**, where the input is usually one structure and the target is one property of that structure. The MGT paper, for example, evaluates the model on MatBench and QMOF tasks and introduces a graph representation with local, line and global/Coulomb-type interactions.

The present task is different in two important ways:

1. The inputs are **isolated molecules generated from SMILES**, not periodic crystal structures.
2. The target depends on **a solute–solvent pair**, not on a single molecular or crystal structure.

These differences require adaptation. In this notebook, ALIGNN and MGT are used as molecular graph encoders inside a pair-prediction architecture. Therefore, underperformance relative to the descriptor pipeline would not necessarily prove that graph neural networks are unsuitable for solvation free energy prediction; it may also reflect the difficulty of adapting architectures designed for single-structure prediction to a paired solute–solvent task.

## Computational note

Both models are expected to require GPU training. This notebook prepares the data, defines the pair-modelling strategy, and analyses the predictions. The actual ALIGNN/MGT training can be run as HPC jobs, and the predicted ΔG_sol values can then be loaded back into the notebook for evaluation.

## Section 1 — Research question and recap of Notebook 2 baselines

Notebook 2 established benchmark performance levels using engineered molecular representations on the ML_Gibbs database of 6,239 solute–solvent pairs.

| Benchmark | Best model | Features | Test MAE | Test RMSE | Test R² |
|---|---|---|---:|---:|---:|
| Random-split test set | SVR-rbf | RDKit descriptors | 0.211 | 0.497 | 0.951 |
| Random-split tree baseline | Random Forest | RDKit descriptors | 0.244 | 0.432 | 0.963 |
| Random-split fingerprint baseline | Random Forest | Morgan fingerprints | 0.469 | 0.853 | 0.857 |
| Water solvent hold-out | Random Forest | RDKit descriptors | 2.809 | 3.786 | 0.031 |

These are the main targets for the graph models. Two comparisons matter most:

1. **Random-split test performance:** can ALIGNN or MGT match the descriptor-based SVR-rbf and Random Forest models?
2. **Water hold-out performance:** can graph models generalise better to an unseen solvent environment, especially water?

If graph models perform better on both the random split and water hold-out, this would suggest that learned molecular representations capture transferable solute–solvent information better than fixed descriptors. If they match the random-split results but still fail on water hold-out, the main limitation may be data coverage rather than representation alone. If they perform worse on both, then engineered RDKit descriptors remain the stronger approach for this dataset size.

In [ ]:
# Section 1 — Fixed-split Notebook 2 baselines for graph-model comparison

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RANDOM_STATE = 42

nb2_fixed_split_baselines = pd.DataFrame([
    {
        "Benchmark": "Best fixed-split classical model by MAE",
        "Model": "SVR-rbf",
        "Representation": "RDKit descriptors",
        "Split protocol": "fixed random 80/10/10 split",
        "Test MAE": 0.2113,
        "Test RMSE": 0.4967,
        "Test R²": 0.9513,
        "Status": "Notebook 2 fixed-split baseline",
    },
    {
        "Benchmark": "Best fixed-split descriptor tree model",
        "Model": "RandomForest",
        "Representation": "RDKit descriptors",
        "Split protocol": "fixed random 80/10/10 split",
        "Test MAE": 0.2442,
        "Test RMSE": 0.4317,
        "Test R²": 0.9632,
        "Status": "Notebook 2 fixed-split baseline",
    },
    {
        "Benchmark": "Best fixed-split fingerprint model",
        "Model": "RandomForest",
        "Representation": "Morgan fingerprints",
        "Split protocol": "fixed random 80/10/10 split",
        "Test MAE": 0.4692,
        "Test RMSE": 0.8526,
        "Test R²": 0.8566,
        "Status": "Notebook 2 fixed-split baseline",
    },
    {
        "Benchmark": "Water solvent hold-out baseline",
        "Model": "RandomForest",
        "Representation": "RDKit descriptors",
        "Split protocol": "water removed from training",
        "Test MAE": 2.8091,
        "Test RMSE": 3.7855,
        "Test R²": 0.0306,
        "Status": "Notebook 2 solvent hold-out baseline",
    },
])

display(nb2_fixed_split_baselines.round(4))

print("Targets for Notebook 3:")
print("Random-split MAE target:", "0.211 kcal/mol for SVR-rbf descriptors")
print("Descriptor tree MAE target:", "0.244 kcal/mol for RandomForest descriptors")
print("Water hold-out MAE target:", "2.809 kcal/mol for RandomForest descriptors")

### Comparison protocol

Notebook 2 now contains two types of classical-model evaluation:

1. A fixed 80/10/10 split, used for direct comparison with ALIGNN and MGT.
2. A 10-fold cross-validation robustness analysis, used to estimate uncertainty in the classical baselines.

For the graph-model comparison in this notebook, the fixed split is the correct comparator because ALIGNN and MGT predictions are produced on the same train/validation/test partition. The cross-validation results from Notebook 2 are discussed only as robustness estimates, not as the direct benchmark for these graph runs.

The water hold-out result from Notebook 2 is kept as a separate extrapolation benchmark. It should only be compared with ALIGNN/MGT once those graph models have also been trained and evaluated under the same water hold-out protocol.

## Section 2 — Dataset loading and split reuse

For a fair comparison with Notebook 2, this notebook uses the same dataset and the same train/validation/test strategy. If the saved split file from Notebook 2 is available, it is loaded directly. Otherwise, the split is recreated using the same random seed and solvent-type stratification used in Notebook 2.

The graph models will be trained on `train_idx`, monitored on `val_idx`, and finally evaluated on `test_idx`.

In [ ]:
# Section 2 — Dataset loading and split reuse

from sklearn.model_selection import train_test_split

df = pd.read_csv("data/ML_Gibbs_Full_Database.csv")
y = df["dG"].to_numpy(dtype=float)

try:
    splits = np.load("split_indices.npz")
    train_idx = splits["train_idx"]
    val_idx = splits["val_idx"]
    test_idx = splits["test_idx"]
    print("Loaded split_indices.npz from disk.")

except FileNotFoundError:
    print("split_indices.npz not found. Recreating the split using the Notebook 2 logic.")

    all_indices = np.arange(len(df))

    train_idx, temp_idx = train_test_split(
        all_indices,
        test_size=0.20,
        random_state=RANDOM_STATE,
        shuffle=True,
        stratify=df["Solvent Type"]
    )

    val_idx, test_idx = train_test_split(
        temp_idx,
        test_size=0.50,
        random_state=RANDOM_STATE,
        shuffle=True,
        stratify=df.loc[temp_idx, "Solvent Type"]
    )

print(f"Dataset: {len(df)} solute–solvent pairs")
print(f"Train / Validation / Test: {len(train_idx)} / {len(val_idx)} / {len(test_idx)}")
print(f"ΔG_sol range: [{y.min():.2f}, {y.max():.2f}] kcal/mol")

# Verify no overlap between subsets
assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0

print("Split partition verified: no overlap between train / validation / test.")

# Check solvent-type balance
for name, idx in [("Train", train_idx), ("Validation", val_idx), ("Test", test_idx)]:
    sub = df.iloc[idx]
    print(f"\n{name} solvent class distribution:")
    print(sub["Solvent Type"].value_counts(normalize=True).mul(100).round(2).to_string())

## Section 3 — Pair-handling strategy: representing a solute–solvent pair

### The problem

ALIGNN and MGT are designed to predict properties from **one graph-structured input**. In contrast, solvation free energy is a property of a **solute–solvent pair**. The same solute can have different ΔG_sol values in different solvents, and the same solvent can give different ΔG_sol values with different solutes. Therefore, both molecules must be represented.

Three possible strategies are considered:

**A. Disjoint single graph.**  
Build one graph containing both molecules as disconnected components and tag each atom with a solute/solvent flag. This is simple, but message passing cannot occur between the two disconnected components.

**B. Solute graph plus solvent features.**  
Encode the solute as a graph and concatenate solvent descriptors or fingerprints to the pooled solute embedding. This is easy to implement, but it treats solute and solvent asymmetrically.

**C. Two-encoder late fusion.**  
Encode the solute and solvent separately using the same graph encoder, concatenate the two embeddings, and pass them through a regression head. This keeps the architecture close to the published ALIGNN/MGT encoders while adapting the output layer to a paired molecular property.

### Decision

This notebook uses **Strategy C: two-encoder late fusion**.

This is the most practical and controlled strategy for comparing ALIGNN and MGT against the descriptor baseline. The same encoder weights are used for solute and solvent, but the concatenation order is fixed as `[solute_embedding, solvent_embedding]`, so the model still knows which molecule plays which role.

The limitation is that solute–solvent interactions are not explicitly represented during message passing. The model must learn the interaction through the final pair head. Therefore, this notebook tests whether ALIGNN/MGT molecular embeddings are strong enough for ΔG_sol prediction, rather than testing a fully explicit solute–solvent interaction graph.

A later extension could add cross-molecule global edges between solute and solvent atoms. That would be more directly aligned with MGT’s long-range/Coulomb interaction idea, but it would require deeper modification of the published architecture.

In [ ]:
# Section 3 — Pair-handling strategy: two-encoder late-fusion head

import torch
import torch.nn as nn

class SolvationPairHead(nn.Module):
    """
    Two-encoder late-fusion architecture.

    The same encoder is applied to the solute graph and the solvent graph.
    The pooled molecular embeddings are concatenated in a fixed order:
    [solute_embedding, solvent_embedding].

    The final MLP predicts ΔG_sol for the solute–solvent pair.
    """
    def __init__(self, encoder: nn.Module, embedding_dim: int, hidden_dim: int = 128):
        super().__init__()

        self.encoder = encoder

        self.head = nn.Sequential(
            nn.Linear(2 * embedding_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, solute_graph, solvent_graph):
        h_solute = self.encoder(solute_graph)
        h_solvent = self.encoder(solvent_graph)

        pair_embedding = torch.cat([h_solute, h_solvent], dim=-1)

        return self.head(pair_embedding).squeeze(-1)


# Dummy encoder only for checking tensor dimensions
class DummyEncoder(nn.Module):
    def __init__(self, embedding_dim=64):
        super().__init__()
        self.embedding_dim = embedding_dim

    def forward(self, graph):
        # In this dummy example, graph is already a batch of embeddings.
        return graph


dummy_encoder = DummyEncoder(embedding_dim=64)
pair_model = SolvationPairHead(
    encoder=dummy_encoder,
    embedding_dim=64,
    hidden_dim=128
)

B = 4
solute_embedding = torch.randn(B, 64)
solvent_embedding = torch.randn(B, 64)

pred = pair_model(solute_embedding, solvent_embedding)

print(f"Output shape: {pred.shape}  expected: torch.Size([{B}])")
print(f"Trainable parameters in pair head: {sum(p.numel() for p in pair_model.head.parameters()):,}")

For both ALIGNN and MGT, the published models will be used as **graph encoders**. Their final regression layers are removed or bypassed, and the pooled molecular embedding is used as the learned representation of each molecule.

Practical implementation plan:

- Generate one graph for each unique solute SMILES and one graph for each unique solvent SMILES.
- Pass the solute graph through the encoder to obtain a solute embedding.
- Pass the solvent graph through the same encoder to obtain a solvent embedding.
- Concatenate the embeddings in fixed order: `[solute, solvent]`.
- Train a small feed-forward head to predict ΔG_sol.

This keeps the comparison focused: Notebook 2 used hand-crafted molecular features, while Notebook 3 tests whether learned graph embeddings can provide an equally strong or stronger molecular representation.

## Section 4 — 3D conformer generation from SMILES

ALIGNN and MGT require graph structures with geometric information such as atom positions, distances and angles. The ML_Gibbs dataset provides SMILES strings, so approximate 3D conformers are generated for each unique solute and solvent molecule.

For each unique canonical SMILES, the molecule is parsed with RDKit, explicit hydrogens are added, a single 3D conformer is generated using ETKDGv3, and the geometry is optimised using MMFF94 when possible. If MMFF parameters are unavailable, UFF is used as a fallback.

The generated conformers are isolated gas-phase approximations. They are not solvated structures and they do not represent conformational ensembles. This is a limitation, but it provides a practical geometry input for testing whether graph neural network representations can compete with the engineered descriptor pipeline from Notebook 2.

The conformers are cached to disk so this step only needs to be run once.

In [ ]:
# Section 4 — 3D conformer generation from SMILES

from rdkit import Chem
from rdkit.Chem import AllChem
from pathlib import Path
import pickle
import time
import pandas as pd
import numpy as np

# Safety fallback in case RANDOM_STATE was not already defined
if "RANDOM_STATE" not in globals():
    RANDOM_STATE = 42

CONFORMER_CACHE = Path("conformers.pkl")
CONFORMER_SUMMARY = Path("conformer_summary.csv")

def canonical_smiles(smiles):
    """
    Return canonical RDKit SMILES, or None if RDKit cannot parse the string.
    """
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    return Chem.MolToSmiles(mol)


def smiles_to_3d_mol(smiles, max_attempts=5, seed=RANDOM_STATE):
    """
    Convert a SMILES string to an RDKit Mol with one 3D conformer.

    Procedure:
    1. Parse SMILES.
    2. Add explicit hydrogens.
    3. Generate one ETKDGv3 conformer.
    4. Optimise with MMFF94 if available, otherwise UFF.
    
    Returns:
    - mol: RDKit Mol with 3D conformer, or None
    - status: string describing success/failure
    """
    base_mol = Chem.MolFromSmiles(str(smiles))
    if base_mol is None:
        return None, "invalid_smiles"

    last_status = "embedding_failed"

    for attempt in range(max_attempts):
        mol = Chem.AddHs(base_mol)

        params = AllChem.ETKDGv3()
        params.randomSeed = int(seed + attempt)

        # Later attempts use random coordinates, which can rescue difficult embeddings.
        if attempt >= 2:
            params.useRandomCoords = True

        try:
            embed_status = AllChem.EmbedMolecule(mol, params)
        except Exception:
            embed_status = -1

        if embed_status != 0:
            continue

        # Force-field optimisation. Failure is not fatal.
        try:
            if AllChem.MMFFHasAllMoleculeParams(mol):
                AllChem.MMFFOptimizeMolecule(mol, maxIters=500)
                return mol, "ok_mmff"
            else:
                AllChem.UFFOptimizeMolecule(mol, maxIters=500)
                return mol, "ok_uff"

        except Exception:
            return mol, "ok_unoptimised"

    return None, last_status


# Collect unique canonical SMILES from both solute and solvent columns
solute_canon = df["Solute SMILES"].apply(canonical_smiles)
solvent_canon = df["Solvent SMILES"].apply(canonical_smiles)

unique_solutes = set(solute_canon.dropna())
unique_solvents = set(solvent_canon.dropna())

all_unique_smiles = sorted(unique_solutes | unique_solvents)

print(f"Unique solute molecules : {len(unique_solutes)}")
print(f"Unique solvent molecules: {len(unique_solvents)}")
print(f"Total unique molecules requiring 3D coordinates: {len(all_unique_smiles)}")


# Load existing cache if available
if CONFORMER_CACHE.exists():
    with open(CONFORMER_CACHE, "rb") as f:
        conformers = pickle.load(f)
    print(f"\nLoaded {len(conformers)} cached conformers from {CONFORMER_CACHE}.")
else:
    conformers = {}
    print("\nNo conformer cache found. Generating conformers from scratch.")


# Generate only missing conformers
missing_to_generate = [smi for smi in all_unique_smiles if smi not in conformers]

print(f"Molecules still requiring conformer generation: {len(missing_to_generate)}")

summary_rows = []
failures = []

if missing_to_generate:
    t0 = time.time()

    for i, smi in enumerate(missing_to_generate, 1):
        mol, status = smiles_to_3d_mol(smi)

        if mol is not None:
            conformers[smi] = mol
        else:
            failures.append(smi)

        summary_rows.append({
            "canonical_smiles": smi,
            "status": status,
            "success": mol is not None,
        })

        if i % 100 == 0 or i == len(missing_to_generate):
            elapsed = time.time() - t0
            print(f"  {i}/{len(missing_to_generate)} processed ({elapsed:.0f}s elapsed)")

    # Save updated cache
    with open(CONFORMER_CACHE, "wb") as f:
        pickle.dump(conformers, f)

    pd.DataFrame(summary_rows).to_csv(CONFORMER_SUMMARY, index=False)

    print(f"\nSaved conformer cache to {CONFORMER_CACHE}.")
    print(f"Saved conformer summary to {CONFORMER_SUMMARY}.")

else:
    print("All conformers already available in cache.")


# Final sanity check
missing = [smi for smi in all_unique_smiles if smi not in conformers]

print(f"\nFinal conformer count: {len(conformers)} / {len(all_unique_smiles)}")

if missing:
    print(f"WARNING: {len(missing)} molecules have no conformer and will be dropped later.")
    print("First missing examples:")
    print(missing[:10])
else:
    print("All dataset molecules have a 3D conformer.")

## Section 5 — Adapting RDKit molecules to ALIGNN / MGT input format

ALIGNN and MGT are designed to consume structure objects with atom species, coordinates and lattice information. RDKit provides isolated molecules with 3D coordinates, but no periodic box. To make the molecules compatible with the graph-building code, each molecule is placed inside a cubic vacuum box.

The box size is chosen dynamically. A minimum 30 Å box is used, but larger molecules are automatically assigned larger boxes so that periodic images remain outside the graph cut-offs. This avoids artificial edges between a molecule and its periodic images.

The output of this section is a dictionary called `structures`, mapping canonical SMILES to `pymatgen.Structure` objects. These structures are later saved in Section 6 for Apocrita training.

In [ ]:
# Section 5 — Convert RDKit Mol → pymatgen.Structure with dynamic vacuum box

from pymatgen.core import Structure, Lattice
import numpy as np

# Cut-offs used by ALIGNN/MGT-style graph construction
LOCAL_CUTOFF = 8.0
GLOBAL_CUTOFF = 12.0

# Minimum vacuum box size. Larger molecules get larger boxes automatically.
MIN_BOX_SIZE = 30.0
BOX_BUFFER = 6.0

def mol_to_structure(
    mol,
    min_box_size=MIN_BOX_SIZE,
    cutoff=GLOBAL_CUTOFF,
    buffer=BOX_BUFFER
):
    """
    Convert an RDKit Mol with a 3D conformer into a pymatgen.Structure.

    The molecule is centred in a cubic vacuum box. The box is enlarged
    automatically for long molecules so that periodic images remain outside
    the graph cut-off.
    """
    if mol is None or mol.GetNumConformers() == 0:
        return None

    conf = mol.GetConformer()

    coords = np.array([
        list(conf.GetAtomPosition(i))
        for i in range(mol.GetNumAtoms())
    ], dtype=float)

    species = [atom.GetSymbol() for atom in mol.GetAtoms()]

    # Centre molecule around zero
    centroid = coords.mean(axis=0)
    centred = coords - centroid

    # Molecular span along each Cartesian axis
    span = centred.max(axis=0) - centred.min(axis=0)
    max_span = float(span.max())

    # Dynamic box size:
    # max molecular span + two cut-off distances + buffer
    box_size = max(
        float(min_box_size),
        max_span + 2.0 * float(cutoff) + float(buffer)
    )

    # Move centred molecule to the middle of the box
    coords_box = centred + box_size / 2.0

    lattice = Lattice.cubic(box_size)

    structure = Structure(
        lattice,
        species,
        coords_box,
        coords_are_cartesian=True
    )

    return structure


def periodic_image_separation(structure):
    """
    Approximate minimum separation between a molecule and its periodic image.
    For a centred molecule in a cubic box, this is approximately:
    box length - maximum molecular span.
    """
    coords = structure.cart_coords
    box = float(structure.lattice.a)

    span = coords.max(axis=0) - coords.min(axis=0)
    max_span = float(span.max())

    return box - max_span


# Build structure cache
structures = {}

for smi, mol in conformers.items():
    structure = mol_to_structure(mol)

    if structure is not None:
        structures[smi] = structure

print(f"Built {len(structures)} pymatgen Structures from {len(conformers)} conformers.")

# Check periodic-image separation for all structures
separations = np.array([
    periodic_image_separation(structure)
    for structure in structures.values()
])

box_sizes = np.array([
    float(structure.lattice.a)
    for structure in structures.values()
])

print(f"\nBox size range: {box_sizes.min():.1f}–{box_sizes.max():.1f} Å")
print(f"Minimum periodic-image separation: {separations.min():.1f} Å")
print(f"Median periodic-image separation : {np.median(separations):.1f} Å")
print(f"Target: should be greater than the global cut-off ({GLOBAL_CUTOFF:.1f} Å).")

if separations.min() <= GLOBAL_CUTOFF:
    print("\nWARNING: At least one structure may see its periodic image.")
    print("Increase MIN_BOX_SIZE or BOX_BUFFER and rerun this section.")
else:
    print("\nPeriodic-image separation check passed.")

# Example structure
example_smi = next(iter(structures))
example_structure = structures[example_smi]

print("\nExample structure:")
print("SMILES:", example_smi)
print("Atoms :", len(example_structure))
print("Box   :", f"{example_structure.lattice.a:.1f} Å")

## Section 6 — Training protocol and input package for Apocrita

The graph models are trained on Apocrita, but this notebook prepares the complete input package.

The training scripts will use:

- `pair_index.csv`: each solute–solvent pair, its split assignment and target value.
- `structures.json`: each unique molecule as a `pymatgen.Structure` dictionary.
- `training_config.json`: shared hyperparameters and target scaling values.

The graph models are trained on `train_idx`, monitored using `val_idx`, and evaluated on `test_idx`.

One asymmetry with Notebook 2 is acknowledged: the final classical baselines in Notebook 2 were refit on `train_idx ∪ val_idx`, while the graph models are trained only on `train_idx` and selected using `val_idx`. This gives the classical baselines slightly more training data, but avoids rerunning expensive multi-hour neural network jobs only to include the validation set. The comparison is therefore conservative for ALIGNN and MGT.

For neural-network stability, ΔG_sol is standardised using the training-set mean and standard deviation. Training can use the scaled target, while all final MAE/RMSE values should be reported after converting predictions back to kcal/mol.

In [ ]:
# Section 6 — Training protocol and Apocrita input files

import json
from pathlib import Path
import pandas as pd
import numpy as np

CONFIG_DIR = Path("notebook3_config")
CONFIG_DIR.mkdir(exist_ok=True)


# 1. Target scaling from training set only

target_mean = float(y[train_idx].mean())
target_std = float(y[train_idx].std())

print(f"Training-set target mean: {target_mean:.4f} kcal/mol")
print(f"Training-set target std : {target_std:.4f} kcal/mol")


# 2. Shared training configuration

TRAINING_CONFIG = {
    "task": "solute_solvent_solvation_free_energy",
    "target": "dG",
    "target_units": "kcal/mol",

    # Scaling
    "target_scaled": True,
    "target_mean_train": target_mean,
    "target_std_train": target_std,

    # Training protocol
    "optimizer": "Adam",
    "learning_rate": 1e-4,
    "weight_decay": 1e-5,
    "loss": "MSE_on_scaled_target",
    "reported_metrics": ["MAE_kcal_mol", "RMSE_kcal_mol", "R2"],

    "max_epochs": 200,
    "early_stopping_metric": "validation_MAE_kcal_mol",
    "early_stopping_patience": 20,
    "batch_size": 16,
    "seed": int(RANDOM_STATE),

    # Geometry / graph parameters
    "local_cutoff_angstrom": LOCAL_CUTOFF,
    "global_cutoff_angstrom": GLOBAL_CUTOFF,
    "min_box_size_angstrom": MIN_BOX_SIZE,
    "box_buffer_angstrom": BOX_BUFFER,

    # Pair strategy
    "pair_strategy": "two_encoder_late_fusion",
    "head_hidden_dim": 128,

    # Encoder graph-embedding width. Both encoders use 256 dimensions:
    # MGT was set to 256 to match the ALIGNN run for a fair comparison.
    "embedding_dim_alignn": 256,
    "embedding_dim_mgt": 256,

    # Methodological note
    "comparison_note": (
        "Notebook 2 classical baselines were refit on train+val before final test evaluation. "
        "Graph models are trained on train only and selected on validation, so the classical "
        "baselines have a small data advantage."
    )
}

with open(CONFIG_DIR / "training_config.json", "w") as f:
    json.dump(TRAINING_CONFIG, f, indent=2)

print(f"\nSaved {CONFIG_DIR / 'training_config.json'}")


# 3. Build pair index

df_pairs = df.copy()
df_pairs["row_idx"] = np.arange(len(df_pairs))

df_pairs["solute_canon"] = df_pairs["Solute SMILES"].apply(canonical_smiles)
df_pairs["solvent_canon"] = df_pairs["Solvent SMILES"].apply(canonical_smiles)

# Split labels
split_map = np.empty(len(df_pairs), dtype="U5")
split_map[train_idx] = "train"
split_map[val_idx] = "val"
split_map[test_idx] = "test"

df_pairs["split"] = split_map

# Add original and scaled targets
df_pairs["dG"] = df_pairs["dG"].astype(float)
df_pairs["dG_scaled"] = (df_pairs["dG"] - target_mean) / target_std

# Keep only rows where both molecules have structures
has_solute_structure = df_pairs["solute_canon"].isin(structures.keys())
has_solvent_structure = df_pairs["solvent_canon"].isin(structures.keys())
usable_mask = has_solute_structure & has_solvent_structure

dropped_rows = int((~usable_mask).sum())

pair_index = df_pairs.loc[usable_mask, [
    "row_idx",
    "split",
    "solute_canon",
    "solvent_canon",
    "dG",
    "dG_scaled",
    "Solvent Type",
    "Database Origin"
]].rename(columns={
    "solute_canon": "solute",
    "solvent_canon": "solvent",
    "Solvent Type": "solvent_type",
    "Database Origin": "database_origin"
}).reset_index(drop=True)

print(f"\nBuilt pair_index with {len(pair_index)} rows.")
print(f"Dropped rows due to missing structures: {dropped_rows}")
print("\nSplit counts:")
print(pair_index["split"].value_counts().to_string())

# Sanity check split sizes if no rows were dropped
if dropped_rows == 0:
    assert pair_index["split"].value_counts()["train"] == len(train_idx)
    assert pair_index["split"].value_counts()["val"] == len(val_idx)
    assert pair_index["split"].value_counts()["test"] == len(test_idx)

pair_index.to_csv(CONFIG_DIR / "pair_index.csv", index=False)
print(f"\nSaved {CONFIG_DIR / 'pair_index.csv'}")


# 4. Save structures as JSON dictionaries

struct_dict = {
    smi: structure.as_dict()
    for smi, structure in structures.items()
}

with open(CONFIG_DIR / "structures.json", "w") as f:
    json.dump(struct_dict, f)

print(f"Saved {CONFIG_DIR / 'structures.json'}")
print(f"Number of unique molecule structures saved: {len(struct_dict)}")


# 5. Save molecule summary table for checking/debugging

molecule_summary = pd.DataFrame({
    "canonical_smiles": list(structures.keys()),
    "n_atoms": [len(s) for s in structures.values()],
    "box_size_angstrom": [float(s.lattice.a) for s in structures.values()],
    "periodic_image_separation_angstrom": [
        periodic_image_separation(s)
        for s in structures.values()
    ]
})

molecule_summary.to_csv(CONFIG_DIR / "molecule_summary.csv", index=False)

print(f"Saved {CONFIG_DIR / 'molecule_summary.csv'}")

# 6. Final summary

print("\nFiles ready for Apocrita:")
for path in [
    CONFIG_DIR / "training_config.json",
    CONFIG_DIR / "pair_index.csv",
    CONFIG_DIR / "structures.json",
    CONFIG_DIR / "molecule_summary.csv",
]:
    print(" -", path)

print("\nNext step:")
print("Copy the entire notebook3_config/ folder to Apocrita before running ALIGNN/MGT training.")

## Section 6b — Entity-disjoint splits for stricter graph-model evaluation

In [ ]:
# Entity-disjoint split builder for ALIGNN / MGT Apocrita runs
# Builds:
#   1. leave-solvents-out
#   2. leave-solutes-out
#   3. solute-scaffold-disjoint
#
# Input expected:
#   notebook3_config/pair_index.csv
#   notebook3_config/training_config.json
#
# Output:
#   notebook3_entity_splits/<split_name>/pair_index.csv
#   notebook3_entity_splits/<split_name>/training_config.json
#   notebook3_entity_splits/<split_name>/split_diagnostics.csv
#   notebook3_entity_splits/<split_name>/balance_by_solvent_type.csv

import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold


RANDOM_STATE = 42

INPUT_DIR = Path("notebook3_config")
OUTPUT_ROOT = Path("notebook3_entity_splits")
OUTPUT_ROOT.mkdir(exist_ok=True)

pair_index = pd.read_csv(INPUT_DIR / "pair_index.csv")

with open(INPUT_DIR / "training_config.json") as f:
    base_config = json.load(f)

required_cols = {
    "row_idx", "solute", "solvent", "dG",
    "solvent_type", "database_origin"
}
missing = required_cols - set(pair_index.columns)
assert not missing, f"pair_index.csv is missing columns: {missing}"

pair_index = pair_index.copy()
pair_index["dG"] = pair_index["dG"].astype(float)

print("Loaded pair_index:", pair_index.shape)
display(pair_index.head())


def solute_scaffold_key(smiles):
    """
    Return a scaffold grouping key for solute-disjoint scaffold splitting.

    For ring-containing molecules, use Bemis-Murcko scaffold.
    For acyclic molecules, RDKit returns an empty scaffold; grouping all
    acyclic molecules together would be too coarse, so we fall back to
    molecule-level canonical SMILES for those cases.
    """
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return f"invalid::{smiles}"

    scaffold = MurckoScaffold.MurckoScaffoldSmiles(
        mol=mol,
        includeChirality=False
    )

    if scaffold:
        return f"scaffold::{scaffold}"

    canon = Chem.MolToSmiles(mol, canonical=True)
    return f"acyclic::{canon}"


def group_three_way_split(df, group_col, test_size=0.10, val_size=0.10, random_state=42):
    """
    Create train/val/test split with no group overlap.

    test_size and val_size are approximate row fractions. Because splitting is
    done by groups, exact row counts may differ from 80/10/10.
    """
    groups = df[group_col].astype(str).to_numpy()
    all_idx = np.arange(len(df))

    outer = GroupShuffleSplit(
        n_splits=1,
        test_size=test_size,
        random_state=random_state
    )

    train_val_idx, test_idx = next(
        outer.split(all_idx, groups=groups)
    )

    # validation fraction relative to remaining train_val rows
    val_relative_size = val_size / (1.0 - test_size)

    inner = GroupShuffleSplit(
        n_splits=1,
        test_size=val_relative_size,
        random_state=random_state
    )

    train_rel_idx, val_rel_idx = next(
        inner.split(
            train_val_idx,
            groups=groups[train_val_idx]
        )
    )

    train_idx = train_val_idx[train_rel_idx]
    val_idx = train_val_idx[val_rel_idx]

    split = np.empty(len(df), dtype=object)
    split[train_idx] = "train"
    split[val_idx] = "val"
    split[test_idx] = "test"

    out = df.copy()
    out["split"] = split

    return out


def duplicate_and_straddle_checks(df, split_name, disjoint_cols):
    """
    Check exact duplicate pairs and group straddling across splits.
    """
    print(f"\n=== Checks: {split_name} ===")

    # Exact duplicate solute-solvent pairs
    pair_dupes = df.duplicated(subset=["solute", "solvent"], keep=False)
    n_pair_dupe_rows = int(pair_dupes.sum())

    if n_pair_dupe_rows:
        dupes = df.loc[pair_dupes, ["solute", "solvent", "split"]].copy()
        pair_split_counts = (
            dupes.groupby(["solute", "solvent"])["split"]
            .nunique()
            .sort_values(ascending=False)
        )
        straddling_pairs = pair_split_counts[pair_split_counts > 1]
        assert len(straddling_pairs) == 0, (
            f"Exact duplicate solute-solvent pairs straddle splits: "
            f"{len(straddling_pairs)}"
        )

    print(f"Duplicate pair rows: {n_pair_dupe_rows}")

    # Entity/group straddling
    for col in disjoint_cols:
        split_counts = (
            df.groupby(col)["split"]
            .nunique()
            .sort_values(ascending=False)
        )
        straddlers = split_counts[split_counts > 1]

        assert len(straddlers) == 0, (
            f"{col} straddles train/val/test in {split_name}: "
            f"{len(straddlers)} groups"
        )

        print(f"No {col} straddling across splits.")


def split_diagnostics(df, split_name, group_col):
    """
    Print and return split balance diagnostics.
    """
    print(f"\n=== Diagnostics: {split_name} ===")

    diag = (
        df.groupby("split")
        .agg(
            n_rows=("row_idx", "size"),
            n_solutes=("solute", "nunique"),
            n_solvents=("solvent", "nunique"),
            n_groups=(group_col, "nunique"),
            dG_mean=("dG", "mean"),
            dG_std=("dG", "std"),
            dG_min=("dG", "min"),
            dG_max=("dG", "max"),
        )
        .reindex(["train", "val", "test"])
    )

    diag["row_fraction"] = diag["n_rows"] / len(df)

    balance_solvent_type = (
        pd.crosstab(df["split"], df["solvent_type"], normalize="index")
        .reindex(["train", "val", "test"])
        .mul(100)
        .round(2)
    )

    balance_origin = (
        pd.crosstab(df["split"], df["database_origin"], normalize="index")
        .reindex(["train", "val", "test"])
        .mul(100)
        .round(2)
    )

    print("\nSplit summary:")
    display(diag.round(4))

    print("\nSolvent-type balance (% within split):")
    display(balance_solvent_type)

    print("\nDatabase-origin balance (% within split):")
    display(balance_origin)

    return diag, balance_solvent_type, balance_origin


def save_entity_split(df, split_name, group_col, disjoint_cols):
    """
    Recompute target scaling from train only, save Apocrita-ready pair_index/config.
    """
    out_dir = OUTPUT_ROOT / split_name
    out_dir.mkdir(parents=True, exist_ok=True)

    df = df.copy()

    # Recompute target scaling using train only
    train_mask = df["split"] == "train"
    target_mean = float(df.loc[train_mask, "dG"].mean())
    target_std = float(df.loc[train_mask, "dG"].std())

    df["dG_scaled"] = (df["dG"] - target_mean) / target_std

    duplicate_and_straddle_checks(df, split_name, disjoint_cols)
    diag, solvent_balance, origin_balance = split_diagnostics(df, split_name, group_col)

    # Save pair index
    df.to_csv(out_dir / "pair_index.csv", index=False)

    # Save diagnostics
    diag.to_csv(out_dir / "split_diagnostics.csv")
    solvent_balance.to_csv(out_dir / "balance_by_solvent_type.csv")
    origin_balance.to_csv(out_dir / "balance_by_database_origin.csv")

    # Update training config
    config = dict(base_config)
    config["split_protocol"] = split_name
    config["entity_disjoint_group_col"] = group_col
    config["entity_disjoint_cols_checked"] = disjoint_cols
    config["target_mean_train"] = target_mean
    config["target_std_train"] = target_std
    config["target_scaled"] = True
    config["notes"] = (
        f"Entity-disjoint split generated from notebook3_config/pair_index.csv. "
        f"Target scaling recomputed from the train subset only. "
        f"Disjoint columns checked: {disjoint_cols}."
    )

    with open(out_dir / "training_config.json", "w") as f:
        json.dump(config, f, indent=2)

    print(f"\nSaved Apocrita-ready files to: {out_dir}")
    print(" - pair_index.csv")
    print(" - training_config.json")
    print(" - split_diagnostics.csv")
    print(" - balance_by_solvent_type.csv")
    print(" - balance_by_database_origin.csv")

    return df, diag



# Build split 1: leave-solvents-out


leave_solvents_out = group_three_way_split(
    pair_index,
    group_col="solvent",
    test_size=0.10,
    val_size=0.10,
    random_state=RANDOM_STATE
)

leave_solvents_out, diag_solvent = save_entity_split(
    leave_solvents_out,
    split_name="leave_solvents_out",
    group_col="solvent",
    disjoint_cols=["solvent"]
)



# Build split 2: leave-solutes-out


leave_solutes_out = group_three_way_split(
    pair_index,
    group_col="solute",
    test_size=0.10,
    val_size=0.10,
    random_state=RANDOM_STATE
)

leave_solutes_out, diag_solute = save_entity_split(
    leave_solutes_out,
    split_name="leave_solutes_out",
    group_col="solute",
    disjoint_cols=["solute"]
)



# Build split 3: solute-scaffold-disjoint


pair_index_scaffold = pair_index.copy()
pair_index_scaffold["solute_scaffold"] = pair_index_scaffold["solute"].apply(solute_scaffold_key)

print("\nScaffold groups:", pair_index_scaffold["solute_scaffold"].nunique())
print("Largest scaffold groups:")
display(
    pair_index_scaffold["solute_scaffold"]
    .value_counts()
    .head(15)
    .to_frame("n_rows")
)

scaffold_disjoint = group_three_way_split(
    pair_index_scaffold,
    group_col="solute_scaffold",
    test_size=0.10,
    val_size=0.10,
    random_state=RANDOM_STATE
)

scaffold_disjoint, diag_scaffold = save_entity_split(
    scaffold_disjoint,
    split_name="scaffold_disjoint",
    group_col="solute_scaffold",
    disjoint_cols=["solute_scaffold"]
)


print("\nDone. Entity-disjoint split folders are ready under:")
print(OUTPUT_ROOT.resolve())

### Software environment and version capture

The cell below records the active software environment. It should be run once in the environment used for notebook execution. GPU details are only expected to appear when the notebook is run inside a GPU-backed session, such as an Apocrita/OnDemand GPU job.

The graph-training jobs themselves were executed on Apocrita/Andrena; their Slurm logs record the corresponding PyTorch, CUDA, DGL and GPU details for the production runs.

In [ ]:
# Reproducibility — software and hardware versions

import sys
import platform
import importlib
import subprocess
from pathlib import Path

print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())

packages = [
    ("numpy", "NumPy"),
    ("pandas", "pandas"),
    ("sklearn", "scikit-learn"),
    ("rdkit", "RDKit"),
    ("pymatgen", "pymatgen"),
    ("torch", "PyTorch"),
    ("dgl", "DGL"),
    ("alignn", "ALIGNN"),
]

for module_name, label in packages:
    try:
        module = importlib.import_module(module_name)
        version = getattr(module, "__version__", "version not exposed")
        print(f"{label}: {version}")
    except ImportError:
        print(f"{label}: not installed/importable in this environment")

try:
    import torch
    print("Torch CUDA:", torch.version.cuda)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("Torch/GPU check failed:", exc)

# Optional: record a local Git commit if the MGT source tree is available.
for candidate in [Path("MGT")]:
    if candidate.exists():
        try:
            commit = subprocess.check_output(
                ["git", "-C", str(candidate), "rev-parse", "HEAD"],
                text=True,
                stderr=subprocess.DEVNULL,
            ).strip()
            print(f"MGT git commit ({candidate}):", commit)
        except Exception:
            print(f"MGT source found at {candidate}, but no Git commit could be read.")

In [ ]:
from pathlib import Path

required_files = [
    "notebook3_config/training_config.json",
    "notebook3_config/pair_index.csv",
    "notebook3_config/structures.json",
    "notebook3_config/molecule_summary.csv",
]

for f in required_files:
    print(f, Path(f).exists())

## Section 7 — MGT data preparation narrative log

This section is a narrative log of the MGT preparation checks completed on Apocrita before the full MGT training jobs. Before training MGT, two Apocrita checks were completed. The first script, `make_mgt_xyz.py`, converted the 949 saved molecular structures into MGT-compatible `.xyz` files and created the required `mgt_data` folder. The script successfully loaded 949 structures, wrote 949 `.xyz` files, generated `id_prop.csv`, copied `atom_init.json`, and confirmed that all elements in the dataset were covered.

The second script, `probe_mgt_graphs.py`, tested whether MGT could build graphs for all molecules, including very small molecules with only 2–3 atoms. The probe successfully loaded all 949 molecular graphs with 0 failures. Several “not enough neighbors” warnings appeared because small molecules cannot provide 12 neighbours, but these were warnings rather than fatal errors.

This confirmed that the MGT graph-construction pipeline was usable for the current molecular dataset before the full MGT jobs were run on Apocrita.

## Section 8 — Full ALIGNN training run on Apocrita

The full ALIGNN job was submitted with `sbatch submit_alignn_andrena.sh` on the `andrena` GPU partition using the `pilot_andrena` account.

The terminal log confirms that job `12878718` ran as `alignn_full`, finished with Slurm state `COMPLETED`, exit code `0:0`, and elapsed time `00:29:18`. The run built graphs for all 949 unique molecules, trained until early stopping at epoch 63, reloaded the best checkpoint, and wrote `predictions_alignn.csv`.

The ALIGNN encoder used a 256-dimensional graph embedding and 4,174,593 trainable parameters (the same architecture as the ALIGNN water hold-out run in Section 10).

The best checkpoint was selected at epoch 43, where the **validation** metrics were MAE = 0.2254 kcal mol$^{-1}$, RMSE = 0.4266 kcal mol$^{-1}$ and R² = 0.9704 (these match the validation split of `predictions_alignn.csv`). The final downloaded outputs are `predictions_alignn.csv` and `alignn_best.pt`; the corresponding Slurm log for this run is `alignn_full.o12878718`, archived in the Apocrita project directory (see the Reproducibility appendix), which verifies the GPU run, graph construction for all 949 molecules, the 4,174,593 trainable parameters, early stopping and the final metrics.

In [ ]:
# Section 8 — Load full ALIGNN predictions and verify run artefacts

from pathlib import Path
import pandas as pd
import numpy as np

full_alignn_candidate_paths = [
    Path("predictions_alignn.csv"),
    Path("predictions_alignn_.csv"),
    Path("results_20260624/predictions_alignn.csv"),
]

for candidate in full_alignn_candidate_paths:
    if candidate.exists():
        full_alignn_pred_path = candidate
        break
else:
    raise FileNotFoundError(
        "Could not find predictions_alignn.csv or predictions_alignn_.csv. Place it beside this notebook "
        "or place it in the repository root."
    )

full_alignn_preds = pd.read_csv(full_alignn_pred_path)

required_cols = {"row_idx", "split", "dG_true", "dG_pred"}
missing_cols = required_cols - set(full_alignn_preds.columns)
assert not missing_cols, f"Missing columns from full ALIGNN predictions: {missing_cols}"
assert len(full_alignn_preds) == 6239, "Full ALIGNN prediction file should contain all 6,239 rows."
assert full_alignn_preds["row_idx"].is_unique, "row_idx values should be unique."
assert full_alignn_preds[["dG_true", "dG_pred"]].notna().all().all(), "Predictions contain missing values."

print("Loaded:", full_alignn_pred_path)
print("Prediction file shape:", full_alignn_preds.shape)
print("Split counts:")
print(full_alignn_preds["split"].value_counts())

display(full_alignn_preds.head())

In [ ]:
# Section 8 — Full ALIGNN metric summary

# Reuse regression_metrics() if it has already been defined; otherwise define it here.
if "regression_metrics" not in globals():
    def regression_metrics(y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)
        mae = np.mean(np.abs(y_true - y_pred))
        rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
        r2 = 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)
        return mae, rmse, r2

full_alignn_rows = []
for split in ["train", "val", "test"]:
    sub = full_alignn_preds[full_alignn_preds["split"] == split]
    mae, rmse, r2 = regression_metrics(sub["dG_true"], sub["dG_pred"])
    full_alignn_rows.append({
        "Model": "Full ALIGNN",
        "Split": split,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2,
        "n": len(sub),
    })

full_alignn_metrics = pd.DataFrame(full_alignn_rows)
display(full_alignn_metrics.round(4))

In [ ]:
# Section 8 — Compare full ALIGNN against Notebook 2 baselines

full_alignn_test = full_alignn_metrics[full_alignn_metrics["Split"] == "test"].iloc[0]

comparison_full_alignn = pd.DataFrame([
    {
        "Model": "SVR-rbf",
        "Representation": "RDKit descriptors",
        "Test MAE": 0.2113,
        "Test RMSE": 0.4967,
        "Test R²": 0.9513,
        "Status": "Notebook 2 fixed-split baseline",
    },
    {
        "Model": "Random Forest",
        "Representation": "RDKit descriptors",
        "Test MAE": 0.2442,
        "Test RMSE": 0.4317,
        "Test R²": 0.9632,
        "Status": "Notebook 2 fixed-split baseline",
    },
    {
        "Model": "Random Forest",
        "Representation": "Morgan fingerprints",
        "Test MAE": 0.4692,
        "Test RMSE": 0.8526,
        "Test R²": 0.8566,
        "Status": "Notebook 2 fixed-split baseline",
    },
    {
        "Model": "Full ALIGNN",
        "Representation": "Learned molecular graphs",
        "Test MAE": full_alignn_test["MAE"],
        "Test RMSE": full_alignn_test["RMSE"],
        "Test R²": full_alignn_test["R²"],
        "Status": "Full Apocrita andrena run, completed",
    },
])

comparison_full_alignn = comparison_full_alignn.sort_values("Test MAE").reset_index(drop=True)
display(comparison_full_alignn.round(4))

In [ ]:
# Section 8 — Full ALIGNN prediction and residual plots

import matplotlib.pyplot as plt

ALIGNN_COLOR = "navy"
IDENTITY_COLOR = "black"

full_test_alignn = full_alignn_preds[full_alignn_preds["split"] == "test"].copy()
full_test_alignn["residual"] = full_test_alignn["dG_pred"] - full_test_alignn["dG_true"]
full_test_alignn["abs_error"] = full_test_alignn["residual"].abs()

plt.figure(figsize=(5, 5))
plt.scatter(
    full_test_alignn["dG_true"],
    full_test_alignn["dG_pred"],
    alpha=0.65,
    color=ALIGNN_COLOR,
    edgecolor="white",
    linewidth=0.3,
)

lims = [
    min(full_test_alignn["dG_true"].min(), full_test_alignn["dG_pred"].min()),
    max(full_test_alignn["dG_true"].max(), full_test_alignn["dG_pred"].max()),
]

plt.plot(lims, lims, linestyle="--", color=IDENTITY_COLOR, linewidth=1)
plt.xlim(lims)
plt.ylim(lims)
plt.gca().set_aspect("equal", adjustable="box")
plt.xlabel("Experimental ΔG$_{sol}$ / kcal mol$^{-1}$")
plt.ylabel("Predicted ΔG$_{sol}$ / kcal mol$^{-1}$")
plt.title("Full ALIGNN: test-set predictions")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.scatter(
    full_test_alignn["dG_true"],
    full_test_alignn["residual"],
    alpha=0.65,
    color=ALIGNN_COLOR,
    edgecolor="white",
    linewidth=0.3,
)
plt.axhline(0, linestyle="--", color=IDENTITY_COLOR, linewidth=1)
plt.xlabel("Experimental ΔG$_{sol}$ / kcal mol$^{-1}$")
plt.ylabel("Residual / kcal mol$^{-1}$")
plt.title("Full ALIGNN residuals on test set")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Largest absolute errors on the full ALIGNN test set:")
display(full_test_alignn.sort_values("abs_error", ascending=False).head(10))

In [ ]:
# Section 8 — Optional full ALIGNN error analysis by solvent type

pair_index_candidates = [
    Path("notebook3_config/pair_index.csv"),
]

for candidate in pair_index_candidates:
    if candidate.exists():
        full_pair_index_path = candidate
        break
else:
    full_pair_index_path = None

if full_pair_index_path is None:
    print("pair_index.csv was not found, so solvent-type error analysis is skipped.")
else:
    full_pair_index = pd.read_csv(full_pair_index_path)
    full_alignn_with_meta = full_alignn_preds.merge(
        full_pair_index[["row_idx", "solute", "solvent", "solvent_type", "database_origin"]],
        on="row_idx",
        how="left",
    )
    full_alignn_with_meta["residual"] = full_alignn_with_meta["dG_pred"] - full_alignn_with_meta["dG_true"]
    full_alignn_with_meta["abs_error"] = full_alignn_with_meta["residual"].abs()
    full_alignn_with_meta["sq_error"] = full_alignn_with_meta["residual"] ** 2

    full_test_meta = full_alignn_with_meta[full_alignn_with_meta["split"] == "test"].copy()
    full_solvent_type_metrics = (
        full_test_meta.groupby("solvent_type")
        .agg(
            n=("dG_true", "size"),
            MAE=("abs_error", "mean"),
            RMSE=("sq_error", lambda x: np.sqrt(np.mean(x))),
            mean_residual=("residual", "mean"),
        )
        .reset_index()
        .sort_values("MAE")
    )

    display(full_solvent_type_metrics.round(4))

    plt.figure(figsize=(9, 4))
    plt.bar(full_solvent_type_metrics["solvent_type"], full_solvent_type_metrics["MAE"])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Test MAE / kcal mol$^{-1}$")
    plt.title("Full ALIGNN test MAE by solvent type")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

### Full ALIGNN conclusion

The full ALIGNN training completed successfully and is the main ALIGNN result for the fixed random split. The run produced predictions for all 6,239 solute–solvent pairs with the intended split sizes: 4,991 train, 624 validation and 624 test rows.

On the test set, full ALIGNN achieved MAE = 0.2050 kcal mol$^{-1}$, RMSE = 0.3090 kcal mol$^{-1}$ and R² = 0.9812. This is stronger than the fixed-split Notebook 2 descriptor baselines on this split. The comparison remains a fixed random-split comparison; it does not replace the separate water hold-out extrapolation experiment.

## Section 9 — Full MGT training run on Apocrita

The full Molecular Graph Transformer (MGT) model was trained on Apocrita/Andrena. The full job `12891378` ran on node `sbg17` with CUDA available on an NVIDIA A100-PCIE-40GB GPU.

The run built and cached 949 molecular graphs with 0 graph-building failures, used the same fixed random split as the ALIGNN comparison, and trained a two-encoder late-fusion MGT model with 3,111,969 trainable parameters. This is fewer parameters than the ALIGNN encoder (4,174,593), so MGT reaches its stronger random-split result with a smaller model. Training stopped by early stopping at epoch 134 after the best validation MAE was reached at epoch 114.

The saved full-run artefacts are:

- `predictions_mgt_full.csv`
- `training_log_mgt_full.csv`
- `mgt_full_best.pt`

The error log contains MGT graph-construction warnings for small or sparse molecular graphs with few neighbours. These warnings are expected for isolated small molecules and did not stop the run; the output log confirms 949 graphs were cached with 0 failures and the job completed successfully.

In [ ]:
# Section 9 — Load full MGT predictions and training log

from pathlib import Path
import pandas as pd
import numpy as np

mgt_prediction_candidates = [
    Path("predictions_mgt_full.csv"),
    Path("predictions_mgt_full_.csv"),
    Path("results_20260624/predictions_mgt_full.csv"),
]

for candidate in mgt_prediction_candidates:
    if candidate.exists():
        full_mgt_pred_path = candidate
        break
else:
    raise FileNotFoundError(
        "Could not find predictions_mgt_full.csv. Put it in the same folder as this notebook "
        "or update mgt_prediction_candidates with the correct path."
    )

mgt_log_candidates = [
    Path("training_log_mgt_full.csv"),
    Path("training_log_mgt_full_.csv"),
    Path("results_20260624/training_log_mgt_full.csv"),
]

for candidate in mgt_log_candidates:
    if candidate.exists():
        full_mgt_log_path = candidate
        break
else:
    raise FileNotFoundError(
        "Could not find training_log_mgt_full.csv. Put it in the same folder as this notebook "
        "or update mgt_log_candidates with the correct path."
    )

full_mgt_preds = pd.read_csv(full_mgt_pred_path)
full_mgt_log = pd.read_csv(full_mgt_log_path)

required_cols = {"row_idx", "split", "dG_true", "dG_pred"}
missing_cols = required_cols - set(full_mgt_preds.columns)
assert not missing_cols, f"Missing columns from full MGT predictions: {missing_cols}"
assert len(full_mgt_preds) == 6239, "Full MGT prediction file should contain all 6,239 rows."
assert full_mgt_preds["row_idx"].is_unique, "row_idx values should be unique."
assert full_mgt_preds[["dG_true", "dG_pred"]].notna().all().all(), "Predictions contain missing values."

print("Loaded predictions:", full_mgt_pred_path)
print("Loaded training log:", full_mgt_log_path)
print("Prediction file shape:", full_mgt_preds.shape)
print("Split counts:")
print(full_mgt_preds["split"].value_counts())

display(full_mgt_preds.head())
display(full_mgt_log.head())

In [ ]:
# Section 9 — Full MGT metric summary

# Reuse regression_metrics() if it has already been defined; otherwise define it here.
if "regression_metrics" not in globals():
    def regression_metrics(y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)
        mae = np.mean(np.abs(y_true - y_pred))
        rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
        r2 = 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)
        return mae, rmse, r2

full_mgt_rows = []
for split in ["train", "val", "test"]:
    sub = full_mgt_preds[full_mgt_preds["split"] == split]
    mae, rmse, r2 = regression_metrics(sub["dG_true"], sub["dG_pred"])
    full_mgt_rows.append({
        "Model": "Full MGT",
        "Split": split,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2,
        "n": len(sub),
    })

full_mgt_metrics = pd.DataFrame(full_mgt_rows)
display(full_mgt_metrics.round(4))

best_mgt_epoch = full_mgt_log.loc[full_mgt_log["val_mae"].idxmin()]
print("Best validation epoch:")
print(best_mgt_epoch)

In [ ]:
# Section 9 — Full MGT training behaviour

import matplotlib.pyplot as plt

display(full_mgt_log.tail())

plt.figure(figsize=(7, 4))
plt.plot(full_mgt_log["epoch"], full_mgt_log["train_loss"], marker="o", markersize=3)
plt.xlabel("Epoch")
plt.ylabel("Training loss, MSE on scaled target")
plt.title("Full MGT training loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(full_mgt_log["epoch"], full_mgt_log["val_mae"], marker="o", markersize=3)
plt.axvline(best_mgt_epoch["epoch"], linestyle="--", color="black", linewidth=1)
plt.xlabel("Epoch")
plt.ylabel("Validation MAE / kcal mol$^{-1}$")
plt.title("Full MGT validation MAE")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Section 9 — Compare full MGT with full ALIGNN and Notebook 2 baselines

full_mgt_test = full_mgt_metrics[full_mgt_metrics["Split"] == "test"].iloc[0]

comparison_full_graphs = pd.DataFrame([
    {
        "Model": "SVR-rbf",
        "Representation": "RDKit descriptors",
        "Test MAE": 0.2113,
        "Test RMSE": 0.4967,
        "Test R²": 0.9513,
        "Status": "Notebook 2 fixed-split baseline",
    },
    {
        "Model": "Random Forest",
        "Representation": "RDKit descriptors",
        "Test MAE": 0.2442,
        "Test RMSE": 0.4317,
        "Test R²": 0.9632,
        "Status": "Notebook 2 fixed-split baseline",
    },
    {
        "Model": "Random Forest",
        "Representation": "Morgan fingerprints",
        "Test MAE": 0.4692,
        "Test RMSE": 0.8526,
        "Test R²": 0.8566,
        "Status": "Notebook 2 fixed-split baseline",
    },
    {
        "Model": "Full MGT",
        "Representation": "Learned molecular graphs",
        "Test MAE": full_mgt_test["MAE"],
        "Test RMSE": full_mgt_test["RMSE"],
        "Test R²": full_mgt_test["R²"],
        "Status": "Full Apocrita andrena run, completed",
    },
])

if "full_alignn_metrics" in globals():
    full_alignn_test = full_alignn_metrics[full_alignn_metrics["Split"] == "test"].iloc[0]
    comparison_full_graphs = pd.concat([
        comparison_full_graphs,
        pd.DataFrame([{
            "Model": "Full ALIGNN",
            "Representation": "Learned molecular graphs",
            "Test MAE": full_alignn_test["MAE"],
            "Test RMSE": full_alignn_test["RMSE"],
            "Test R²": full_alignn_test["R²"],
            "Status": "Full Apocrita andrena run, completed",
        }])
    ], ignore_index=True)

comparison_full_graphs = comparison_full_graphs.sort_values("Test MAE").reset_index(drop=True)
display(comparison_full_graphs.round(4))

In [ ]:
# Section 9 — Full MGT prediction and residual plots

MGT_COLOR = "#16a085"        
IDENTITY_COLOR = "black"

# Use plotting helpers if already defined; otherwise define compact local versions.
if "prepare_prediction_subset" not in globals():
    def prepare_prediction_subset(preds, split="test"):
        sub = preds[preds["split"] == split].copy()
        sub["residual"] = sub["dG_pred"] - sub["dG_true"]
        sub["abs_error"] = sub["residual"].abs()
        sub["sq_error"] = sub["residual"] ** 2
        return sub

def plot_true_vs_pred_mgt(sub, title="True vs predicted"):
    plt.figure(figsize=(5, 5))

    plt.scatter(
        sub["dG_true"],
        sub["dG_pred"],
        alpha=0.65,
        color=MGT_COLOR,
        edgecolor="white",
        linewidth=0.3,
    )

    lims = [
        min(sub["dG_true"].min(), sub["dG_pred"].min()),
        max(sub["dG_true"].max(), sub["dG_pred"].max()),
    ]

    plt.plot(lims, lims, linestyle="--", color=IDENTITY_COLOR, linewidth=1)
    plt.xlim(lims)
    plt.ylim(lims)
    plt.gca().set_aspect("equal", adjustable="box")

    plt.xlabel("Experimental ΔG$_{sol}$ / kcal mol$^{-1}$")
    plt.ylabel("Predicted ΔG$_{sol}$ / kcal mol$^{-1}$")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_residuals_mgt(sub, title="Residuals"):
    plt.figure(figsize=(7, 4))

    plt.scatter(
        sub["dG_true"],
        sub["residual"],
        alpha=0.65,
        color=MGT_COLOR,
        edgecolor="white",
        linewidth=0.3,
    )

    plt.axhline(0, linestyle="--", color=IDENTITY_COLOR, linewidth=1)

    plt.xlabel("Experimental ΔG$_{sol}$ / kcal mol$^{-1}$")
    plt.ylabel("Residual / kcal mol$^{-1}$")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_error_distribution_mgt(sub, title="Absolute error distribution"):
    plt.figure(figsize=(7, 4))

    plt.hist(
        sub["abs_error"],
        bins=30,
        color=MGT_COLOR,
        edgecolor="black",
        alpha=0.8,
    )

    plt.xlabel("Absolute error / kcal mol$^{-1}$")
    plt.ylabel("Number of examples")
    plt.title(title)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


def show_outliers_mgt(sub, n=15, title="Largest absolute errors"):
    outliers = sub.sort_values("abs_error", ascending=False).head(n).copy()

    display(outliers)

    plt.figure(figsize=(8, 4))

    plt.bar(
        range(len(outliers)),
        outliers["abs_error"],
        color=MGT_COLOR,
        edgecolor="black",
        alpha=0.85,
    )

    plt.xticks(range(len(outliers)), outliers["row_idx"], rotation=45, ha="right")
    plt.xlabel("Dataset row index")
    plt.ylabel("Absolute error / kcal mol$^{-1}$")
    plt.title(title)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

    return outliers


full_test_mgt = prepare_prediction_subset(full_mgt_preds, split="test")

plot_true_vs_pred_mgt(full_test_mgt, title="Full MGT: test-set predictions")
plot_residuals_mgt(full_test_mgt, title="Full MGT residuals on test set")
plot_error_distribution_mgt(full_test_mgt, title="Full MGT absolute error distribution")
full_mgt_outliers = show_outliers_mgt(
    full_test_mgt,
    n=15,
    title="Largest full MGT test-set errors",
)

In [ ]:
# Section 9 — Optional full MGT error analysis by solvent type

if "plot_solvent_type_errors" in globals():
    full_mgt_solvent_type_metrics = plot_solvent_type_errors(
        full_mgt_preds,
        pair_index_path="notebook3_config/pair_index.csv",
        split="test",
        model_name="Full MGT",
    )
else:
    pair_index_path = Path("notebook3_config/pair_index.csv")
    if not pair_index_path.exists():
        print(f"{pair_index_path} not found. Solvent-type analysis skipped.")
        full_mgt_solvent_type_metrics = None
    else:
        pair_index = pd.read_csv(pair_index_path)
        full_mgt_with_meta = full_mgt_preds.merge(
            pair_index[["row_idx", "solute", "solvent", "solvent_type", "database_origin"]],
            on="row_idx",
            how="left",
        )
        full_mgt_test_meta = prepare_prediction_subset(full_mgt_with_meta, split="test")
        full_mgt_solvent_type_metrics = (
            full_mgt_test_meta.groupby("solvent_type")
            .agg(
                n=("dG_true", "size"),
                MAE=("abs_error", "mean"),
                RMSE=("sq_error", lambda x: np.sqrt(np.mean(x))),
                mean_residual=("residual", "mean"),
            )
            .reset_index()
            .sort_values("MAE")
        )
        display(full_mgt_solvent_type_metrics.round(4))

        plt.figure(figsize=(9, 4))
        plt.bar(full_mgt_solvent_type_metrics["solvent_type"], full_mgt_solvent_type_metrics["MAE"])
        plt.xticks(rotation=45, ha="right")
        plt.ylabel("Test MAE / kcal mol$^{-1}$")
        plt.title("Full MGT test MAE by solvent type")
        plt.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.show()

### Full MGT conclusion

The full MGT model completed successfully and is the strongest fixed random-split graph result in this notebook. It produced predictions for all 6,239 solute–solvent pairs with the intended split sizes: 4,991 train, 624 validation and 624 test rows.

On the test set, full MGT achieved MAE = 0.1532 kcal mol$^{-1}$, RMSE = 0.2830 kcal mol$^{-1}$ and R² = 0.9842. This improves on the full ALIGNN run and the fixed-split Notebook 2 descriptor baselines for this random-split comparison.

As with full ALIGNN, this is still a fixed random-split result. It should not be interpreted as a water hold-out or solvent hold-out extrapolation result.

## Section 10 — Full ALIGNN water hold-out results

The ALIGNN water hold-out job has now completed on Apocrita/Andrena. This is a different experiment from the fixed random split used above: all pairs whose **solvent** is water (`solvent == "O"`, `solvent_type == "Water"`) are excluded from training and validation and used only as the test set. Water may still appear as a *solute* in training pairs; only water-as-solvent pairs are held out.

The downloaded job log confirms that job `12973789` ran on node `sbg16` with CUDA available on an NVIDIA A100-PCIE-40GB GPU. The model built graphs for all 949 unique molecules, trained with early stopping, reloaded the best checkpoint, and saved `predictions_alignn_waterholdout.csv`.

The hold-out split contains 5,037 train rows, 560 validation rows and 642 test rows. The test set is entirely water (`solvent == "O"`, `solvent_type == "Water"`), with zero water rows in train or validation.

In [ ]:
# Section 10 — Load ALIGNN water hold-out predictions, split file and config

from pathlib import Path
import json
import pandas as pd
import numpy as np

water_pred_candidates = [
    Path("predictions_alignn_waterholdout.csv"),
    Path("predictions_alignn_waterholdout_.csv"),
]

for candidate in water_pred_candidates:
    if candidate.exists():
        alignn_water_pred_path = candidate
        break
else:
    raise FileNotFoundError(
        "Could not find predictions_alignn_waterholdout.csv. Put it in the same folder as this notebook "
        "or update water_pred_candidates with the correct path."
    )

water_pair_candidates = [
    Path("pair_index_waterholdout.csv"),
    Path("pair_index_waterholdout_.csv"),
]

for candidate in water_pair_candidates:
    if candidate.exists():
        alignn_water_pair_path = candidate
        break
else:
    raise FileNotFoundError(
        "Could not find pair_index_waterholdout.csv. Put it in the same folder as this notebook "
        "or update water_pair_candidates with the correct path."
    )

water_config_candidates = [
    Path("training_config_waterholdout.json"),
    Path("training_config_waterholdout_.json"),
]

alignn_water_config_path = None
for candidate in water_config_candidates:
    if candidate.exists():
        alignn_water_config_path = candidate
        break

alignn_water_preds = pd.read_csv(alignn_water_pred_path)
alignn_water_pair_index = pd.read_csv(alignn_water_pair_path)

if alignn_water_config_path is not None:
    with open(alignn_water_config_path) as f:
        alignn_water_config = json.load(f)
else:
    alignn_water_config = {}

required_cols = {"row_idx", "split", "dG_true", "dG_pred"}
missing_cols = required_cols - set(alignn_water_preds.columns)
assert not missing_cols, f"Missing columns from water hold-out predictions: {missing_cols}"
assert len(alignn_water_preds) == 6239, "Prediction file should contain all 6,239 rows."
assert alignn_water_preds["row_idx"].is_unique, "row_idx values should be unique."
assert alignn_water_preds[["dG_true", "dG_pred"]].notna().all().all(), "Predictions contain missing values."

print("Loaded predictions:", alignn_water_pred_path)
print("Loaded pair index:", alignn_water_pair_path)
if alignn_water_config_path is not None:
    print("Loaded config:", alignn_water_config_path)

print("Prediction file shape:", alignn_water_preds.shape)
print("Prediction split counts:")
print(alignn_water_preds["split"].value_counts())

display(alignn_water_preds.head())
display(alignn_water_pair_index.head())

In [ ]:
# Section 10 — Verify that this is a true water hold-out split

water_meta = alignn_water_preds.merge(
    alignn_water_pair_index[["row_idx", "solute", "solvent", "solvent_type", "database_origin"]],
    on="row_idx",
    how="left",
)

assert water_meta[["solute", "solvent", "solvent_type"]].notna().all().all(), "Missing metadata after merge."

water_test_meta = water_meta[water_meta["split"] == "test"].copy()
water_train_meta = water_meta[water_meta["split"] == "train"].copy()
water_val_meta = water_meta[water_meta["split"] == "val"].copy()

assert len(water_test_meta) == 642, "Expected 642 water hold-out test rows."
assert (water_test_meta["solvent"] == "O").all(), "All test rows should have water solvent SMILES O."
assert (water_test_meta["solvent_type"] == "Water").all(), "All test rows should be solvent_type Water."
assert not (water_train_meta["solvent"] == "O").any(), "Water rows found in training set."
assert not (water_val_meta["solvent"] == "O").any(), "Water rows found in validation set."

print("Water hold-out verification passed.")
print("Train rows:", len(water_train_meta))
print("Validation rows:", len(water_val_meta))
print("Water test rows:", len(water_test_meta))
print("Test solvent counts:")
print(water_test_meta["solvent_type"].value_counts())

In [ ]:
# Section 10 — ALIGNN water hold-out metric summary

if "regression_metrics" not in globals():
    def regression_metrics(y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)
        mae = np.mean(np.abs(y_true - y_pred))
        rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
        r2 = 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)
        return mae, rmse, r2

alignn_water_rows = []
for split in ["train", "val", "test"]:
    sub = alignn_water_preds[alignn_water_preds["split"] == split]
    mae, rmse, r2 = regression_metrics(sub["dG_true"], sub["dG_pred"])
    alignn_water_rows.append({
        "Model": "ALIGNN water hold-out",
        "Split": split,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2,
        "n": len(sub),
    })

alignn_water_metrics = pd.DataFrame(alignn_water_rows)
display(alignn_water_metrics.round(4))

alignn_water_test = alignn_water_metrics[alignn_water_metrics["Split"] == "test"].iloc[0]

In [ ]:
# Section 10 — Compare ALIGNN water hold-out with Notebook 2 water hold-out baseline

water_holdout_comparison = pd.DataFrame([
    {
        "Model": "Random Forest",
        "Representation": "RDKit descriptors",
        "Test solvent": "water",
        "Test MAE": 2.809,
        "Test RMSE": 3.786,
        "Test R²": 0.031,
        "Status": "Notebook 2 water hold-out baseline",
    },
    {
        "Model": "ALIGNN",
        "Representation": "Learned molecular graphs",
        "Test solvent": "water",
        "Test MAE": alignn_water_test["MAE"],
        "Test RMSE": alignn_water_test["RMSE"],
        "Test R²": alignn_water_test["R²"],
        "Status": "Full Apocrita andrena water hold-out run",
    },
])

water_holdout_comparison = water_holdout_comparison.sort_values("Test MAE").reset_index(drop=True)
display(water_holdout_comparison.round(4))

In [ ]:
# Section 10 — ALIGNN water hold-out prediction and residual plots

import matplotlib.pyplot as plt

WATER_ALIGNN_COLOR = 'skyblue'
IDENTITY_COLOR = "black"

water_test = water_test_meta.copy()
water_test["residual"] = water_test["dG_pred"] - water_test["dG_true"]
water_test["abs_error"] = water_test["residual"].abs()
water_test["sq_error"] = water_test["residual"] ** 2

plt.figure(figsize=(5, 5))
plt.scatter(
    water_test["dG_true"],
    water_test["dG_pred"],
    alpha=0.65,
    color=WATER_ALIGNN_COLOR,
    edgecolor="white",
    linewidth=0.3,
)
lims = [
    min(water_test["dG_true"].min(), water_test["dG_pred"].min()),
    max(water_test["dG_true"].max(), water_test["dG_pred"].max()),
]
plt.plot(lims, lims, linestyle="--", color=IDENTITY_COLOR, linewidth=1)
plt.xlim(lims)
plt.ylim(lims)
plt.gca().set_aspect("equal", adjustable="box")
plt.xlabel("Experimental ΔG$_{sol}$ / kcal mol$^{-1}$")
plt.ylabel("Predicted ΔG$_{sol}$ / kcal mol$^{-1}$")
plt.title("ALIGNN water hold-out: test-set predictions")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.scatter(
    water_test["dG_true"],
    water_test["residual"],
    alpha=0.65,
    color=WATER_ALIGNN_COLOR,
    edgecolor="white",
    linewidth=0.3,
)
plt.axhline(0, linestyle="--", color=IDENTITY_COLOR, linewidth=1)
plt.xlabel("Experimental ΔG$_{sol}$ / kcal mol$^{-1}$")
plt.ylabel("Residual / kcal mol$^{-1}$")
plt.title("ALIGNN water hold-out residuals")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.hist(
    water_test["abs_error"],
    bins=30,
    color=WATER_ALIGNN_COLOR,
    edgecolor="black",
    alpha=0.8,
)
plt.xlabel("Absolute error / kcal mol$^{-1}$")
plt.ylabel("Number of water test examples")
plt.title("ALIGNN water hold-out absolute error distribution")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Section 10 — Largest ALIGNN water hold-out outliers

water_outliers = water_test.sort_values("abs_error", ascending=False).head(15).copy()

display(water_outliers[[
    "row_idx", "solute", "solvent", "solvent_type", "database_origin",
    "dG_true", "dG_pred", "residual", "abs_error"
]])

plt.figure(figsize=(8, 4))
plt.bar(range(len(water_outliers)), water_outliers["abs_error"])
plt.xticks(range(len(water_outliers)), water_outliers["row_idx"], rotation=45, ha="right")
plt.xlabel("Dataset row index")
plt.ylabel("Absolute error / kcal mol$^{-1}$")
plt.title("Largest ALIGNN water hold-out errors")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### ALIGNN water hold-out conclusion

The ALIGNN water hold-out experiment completed successfully and verifies the intended extrapolation split: all 642 water examples were withheld from training and validation and used only for testing.

The model fits the non-water train and validation sets well, but performance drops substantially on the unseen water solvent. The water hold-out test metrics are MAE = 2.7131 kcal mol$^{-1}$, RMSE = 3.3552 kcal mol$^{-1}$ and R² = 0.2385. This confirms that water extrapolation remains much harder than the fixed random split.

Compared with the Notebook 2 descriptor Random Forest water hold-out baseline (MAE = 2.809 kcal mol$^{-1}$, RMSE = 3.786 kcal mol$^{-1}$, R² = 0.031), ALIGNN is slightly better on this extrapolation test, but both models show that water hold-out is the most difficult evaluation setting in the project.

## Section 11 — Full MGT water hold-out results

The full MGT water hold-out job has also completed on Apocrita/Andrena. This uses the same water hold-out split as the ALIGNN water hold-out run in Section 10: water examples are excluded from training and validation, then used as the held-out test solvent.

The downloaded job log confirms that job `13005466` ran on node `sbg15` with CUDA available on an NVIDIA A100-PCIE-40GB GPU. The run built and cached all 949 molecular graphs with 0 graph-building failures, trained a 3,111,969-parameter two-encoder MGT model, stopped by early stopping at epoch 112, and saved `predictions_mgt_waterholdout.csv`, `training_log_mgt_waterholdout.csv` and `mgt_waterholdout_best.pt`.

The error log contains the same MGT neighbour-count warnings seen in the previous MGT runs. These warnings are expected for some small isolated molecules and did not stop the training or prediction workflow.

In [ ]:
# Section 11 — Load MGT water hold-out predictions and training log

from pathlib import Path
import pandas as pd
import numpy as np

mgt_water_pred_candidates = [
    Path("predictions_mgt_waterholdout.csv"),
    Path("predictions_mgt_waterholdout_.csv"),
]

for candidate in mgt_water_pred_candidates:
    if candidate.exists():
        mgt_water_pred_path = candidate
        break
else:
    raise FileNotFoundError(
        "Could not find predictions_mgt_waterholdout.csv. Put it in the same folder as this notebook "
        "or update mgt_water_pred_candidates with the correct path."
    )

mgt_water_log_candidates = [
    Path("training_log_mgt_waterholdout.csv"),
    Path("training_log_mgt_waterholdout_.csv"),
]

for candidate in mgt_water_log_candidates:
    if candidate.exists():
        mgt_water_log_path = candidate
        break
else:
    raise FileNotFoundError(
        "Could not find training_log_mgt_waterholdout.csv. Put it in the same folder as this notebook "
        "or update mgt_water_log_candidates with the correct path."
    )

# Reuse the water hold-out pair index from Section 10 if it exists; otherwise load it here.
if "alignn_water_pair_index" in globals():
    mgt_water_pair_index = alignn_water_pair_index.copy()
else:
    pair_candidates = [
        Path("pair_index_waterholdout.csv"),
        Path("pair_index_waterholdout_.csv"),
        ]
    for candidate in pair_candidates:
        if candidate.exists():
            mgt_water_pair_path = candidate
            break
    else:
        raise FileNotFoundError(
            "Could not find pair_index_waterholdout.csv. Put it in the same folder as this notebook."
        )
    mgt_water_pair_index = pd.read_csv(mgt_water_pair_path)

mgt_water_preds = pd.read_csv(mgt_water_pred_path)
mgt_water_log = pd.read_csv(mgt_water_log_path)

required_cols = {"row_idx", "split", "dG_true", "dG_pred"}
missing_cols = required_cols - set(mgt_water_preds.columns)
assert not missing_cols, f"Missing columns from MGT water hold-out predictions: {missing_cols}"
assert len(mgt_water_preds) == 6239, "Prediction file should contain all 6,239 rows."
assert mgt_water_preds["row_idx"].is_unique, "row_idx values should be unique."
assert mgt_water_preds[["dG_true", "dG_pred"]].notna().all().all(), "Predictions contain missing values."

print("Loaded predictions:", mgt_water_pred_path)
print("Loaded training log:", mgt_water_log_path)
print("Prediction file shape:", mgt_water_preds.shape)
print("Prediction split counts:")
print(mgt_water_preds["split"].value_counts())

display(mgt_water_preds.head())
display(mgt_water_log.head())

In [ ]:
# Section 11 — Verify that MGT used the same true water hold-out split

mgt_water_meta = mgt_water_preds.merge(
    mgt_water_pair_index[["row_idx", "solute", "solvent", "solvent_type", "database_origin"]],
    on="row_idx",
    how="left",
)

assert mgt_water_meta[["solute", "solvent", "solvent_type"]].notna().all().all(), "Missing metadata after merge."

mgt_water_test_meta = mgt_water_meta[mgt_water_meta["split"] == "test"].copy()
mgt_water_train_meta = mgt_water_meta[mgt_water_meta["split"] == "train"].copy()
mgt_water_val_meta = mgt_water_meta[mgt_water_meta["split"] == "val"].copy()

assert len(mgt_water_test_meta) == 642, "Expected 642 water hold-out test rows."
assert (mgt_water_test_meta["solvent"] == "O").all(), "All test rows should have water solvent SMILES O."
assert (mgt_water_test_meta["solvent_type"] == "Water").all(), "All test rows should be solvent_type Water."
assert not (mgt_water_train_meta["solvent"] == "O").any(), "Water rows found in MGT training set."
assert not (mgt_water_val_meta["solvent"] == "O").any(), "Water rows found in MGT validation set."

print("MGT water hold-out verification passed.")
print("Train rows:", len(mgt_water_train_meta))
print("Validation rows:", len(mgt_water_val_meta))
print("Water test rows:", len(mgt_water_test_meta))
print("Test solvent counts:")
print(mgt_water_test_meta["solvent_type"].value_counts())

In [ ]:
# Section 11 — MGT water hold-out metric summary

if "regression_metrics" not in globals():
    def regression_metrics(y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)
        mae = np.mean(np.abs(y_true - y_pred))
        rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
        r2 = 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)
        return mae, rmse, r2

mgt_water_rows = []
for split in ["train", "val", "test"]:
    sub = mgt_water_preds[mgt_water_preds["split"] == split]
    mae, rmse, r2 = regression_metrics(sub["dG_true"], sub["dG_pred"])
    mgt_water_rows.append({
        "Model": "MGT water hold-out",
        "Split": split,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2,
        "n": len(sub),
    })

mgt_water_metrics = pd.DataFrame(mgt_water_rows)
display(mgt_water_metrics.round(4))

mgt_water_test = mgt_water_metrics[mgt_water_metrics["Split"] == "test"].iloc[0]

best_mgt_water_epoch = mgt_water_log.loc[mgt_water_log["val_mae"].idxmin()]
print("Best validation epoch:")
print(best_mgt_water_epoch)

In [ ]:
# Section 11 — MGT water hold-out training behaviour

import matplotlib.pyplot as plt

display(mgt_water_log.tail())

plt.figure(figsize=(7, 4))
plt.plot(mgt_water_log["epoch"], mgt_water_log["train_loss"], marker="o", markersize=3)
plt.xlabel("Epoch")
plt.ylabel("Training loss, MSE on scaled target")
plt.title("MGT water hold-out training loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(mgt_water_log["epoch"], mgt_water_log["val_mae"], marker="o", markersize=3)
plt.axvline(best_mgt_water_epoch["epoch"], linestyle="--", color="black", linewidth=1)
plt.xlabel("Epoch")
plt.ylabel("Validation MAE / kcal mol$^{-1}$")
plt.title("MGT water hold-out validation MAE")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Section 11 — Compare water hold-out models

water_holdout_graph_comparison = pd.DataFrame([
    {
        "Model": "Random Forest",
        "Representation": "RDKit descriptors",
        "Test solvent": "water",
        "Test MAE": 2.809,
        "Test RMSE": 3.786,
        "Test R²": 0.031,
        "Status": "Notebook 2 water hold-out baseline",
    },
    {
        "Model": "MGT",
        "Representation": "Learned molecular graphs",
        "Test solvent": "water",
        "Test MAE": mgt_water_test["MAE"],
        "Test RMSE": mgt_water_test["RMSE"],
        "Test R²": mgt_water_test["R²"],
        "Status": "Full Apocrita andrena water hold-out run",
    },
])

if "alignn_water_test" in globals():
    water_holdout_graph_comparison = pd.concat([
        water_holdout_graph_comparison,
        pd.DataFrame([{
            "Model": "ALIGNN",
            "Representation": "Learned molecular graphs",
            "Test solvent": "water",
            "Test MAE": alignn_water_test["MAE"],
            "Test RMSE": alignn_water_test["RMSE"],
            "Test R²": alignn_water_test["R²"],
            "Status": "Full Apocrita andrena water hold-out run",
        }])
    ], ignore_index=True)

water_holdout_graph_comparison = water_holdout_graph_comparison.sort_values("Test MAE").reset_index(drop=True)
display(water_holdout_graph_comparison.round(4))

In [ ]:
# Section 11 — MGT water hold-out prediction and residual plots

MGT_WATER_COLOR = "salmon"
IDENTITY_COLOR = "black"

mgt_water_test_plot = mgt_water_test_meta.copy()
mgt_water_test_plot["residual"] = mgt_water_test_plot["dG_pred"] - mgt_water_test_plot["dG_true"]
mgt_water_test_plot["abs_error"] = mgt_water_test_plot["residual"].abs()
mgt_water_test_plot["sq_error"] = mgt_water_test_plot["residual"] ** 2

plt.figure(figsize=(5, 5))
plt.scatter(
    mgt_water_test_plot["dG_true"],
    mgt_water_test_plot["dG_pred"],
    alpha=0.65,
    color=MGT_WATER_COLOR,
    edgecolor="white",
    linewidth=0.3,
)
lims = [
    min(mgt_water_test_plot["dG_true"].min(), mgt_water_test_plot["dG_pred"].min()),
    max(mgt_water_test_plot["dG_true"].max(), mgt_water_test_plot["dG_pred"].max()),
]
plt.plot(lims, lims, linestyle="--", color=IDENTITY_COLOR, linewidth=1)
plt.xlim(lims)
plt.ylim(lims)
plt.gca().set_aspect("equal", adjustable="box")
plt.xlabel("Experimental ΔG$_{sol}$ / kcal mol$^{-1}$")
plt.ylabel("Predicted ΔG$_{sol}$ / kcal mol$^{-1}$")
plt.title("MGT water hold-out: test-set predictions")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.scatter(
    mgt_water_test_plot["dG_true"],
    mgt_water_test_plot["residual"],
    alpha=0.65,
    color=MGT_WATER_COLOR,
    edgecolor="white",
    linewidth=0.3,
)
plt.axhline(0, linestyle="--", color=IDENTITY_COLOR, linewidth=1)
plt.xlabel("Experimental ΔG$_{sol}$ / kcal mol$^{-1}$")
plt.ylabel("Residual / kcal mol$^{-1}$")
plt.title("MGT water hold-out residuals")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.hist(
    mgt_water_test_plot["abs_error"],
    bins=30,
    color=MGT_WATER_COLOR,
    edgecolor="black",
    alpha=0.8,
)
plt.xlabel("Absolute error / kcal mol$^{-1}$")
plt.ylabel("Number of water test examples")
plt.title("MGT water hold-out absolute error distribution")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Section 11 — Largest MGT water hold-out outliers

mgt_water_outliers = mgt_water_test_plot.sort_values("abs_error", ascending=False).head(15).copy()

display(mgt_water_outliers[[
    "row_idx", "solute", "solvent", "solvent_type", "database_origin",
    "dG_true", "dG_pred", "residual", "abs_error"
]])

plt.figure(figsize=(8, 4))
plt.bar(range(len(mgt_water_outliers)), mgt_water_outliers["abs_error"])
plt.xticks(range(len(mgt_water_outliers)), mgt_water_outliers["row_idx"], rotation=45, ha="right")
plt.xlabel("Dataset row index")
plt.ylabel("Absolute error / kcal mol$^{-1}$")
plt.title("Largest MGT water hold-out errors")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### MGT water hold-out conclusion

The MGT water hold-out experiment completed successfully and used the intended water-only test set. Like ALIGNN, MGT fits the non-water train and validation data well, but performance drops sharply when extrapolating to water.

The MGT water hold-out test metrics are MAE = 2.8657 kcal mol$^{-1}$, RMSE = 3.4193 kcal mol$^{-1}$ and R² = 0.2091. This is close to, but slightly worse than, the ALIGNN water hold-out result by MAE. It is also slightly worse than the Notebook 2 descriptor Random Forest water hold-out baseline by MAE, although its RMSE and R² are better than that baseline.

Together, the ALIGNN and MGT water hold-out runs show that the learned graph models perform very strongly on the fixed random split, but water remains the most difficult extrapolation setting.

## Section 12 — Final ALIGNN vs MGT comparison

The table below compares the two graph models using only the completed full-training runs. Two evaluation settings are reported separately: the fixed random split and the water hold-out extrapolation test.

In [ ]:
# Section 12 — Final comparison table for the two full graph models

from pathlib import Path
import pandas as pd
import numpy as np

if "regression_metrics" not in globals():
    def regression_metrics(y_true, y_pred):
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)
        mae = np.mean(np.abs(y_true - y_pred))
        rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
        r2 = 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)
        return mae, rmse, r2


def _find_first_existing(candidates):
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate.exists():
            return candidate
    return None


def _test_metrics_from_predictions(candidates):
    path = _find_first_existing(candidates)
    if path is None:
        return None
    preds = pd.read_csv(path)
    sub = preds[preds["split"] == "test"]
    mae, rmse, r2 = regression_metrics(sub["dG_true"], sub["dG_pred"])
    return mae, rmse, r2, len(sub), path

comparison_rows = []

# Fixed random split: ALIGNN
if "full_alignn_metrics" in globals():
    row = full_alignn_metrics[full_alignn_metrics["Split"] == "test"].iloc[0]
    comparison_rows.append({
        "Evaluation": "Fixed random split",
        "Model": "ALIGNN",
        "Test MAE": row["MAE"],
        "Test RMSE": row["RMSE"],
        "Test R²": row["R²"],
        "n_test": int(row["n"]),
    })
else:
    result = _test_metrics_from_predictions(["predictions_alignn.csv", "predictions_alignn_.csv"])
    if result is not None:
        mae, rmse, r2, n, path = result
        comparison_rows.append({"Evaluation": "Fixed random split", "Model": "ALIGNN", "Test MAE": mae, "Test RMSE": rmse, "Test R²": r2, "n_test": n})

# Fixed random split: MGT
if "full_mgt_metrics" in globals():
    row = full_mgt_metrics[full_mgt_metrics["Split"] == "test"].iloc[0]
    comparison_rows.append({
        "Evaluation": "Fixed random split",
        "Model": "MGT",
        "Test MAE": row["MAE"],
        "Test RMSE": row["RMSE"],
        "Test R²": row["R²"],
        "n_test": int(row["n"]),
    })
else:
    result = _test_metrics_from_predictions(["predictions_mgt_full.csv", "predictions_mgt_full_.csv"])
    if result is not None:
        mae, rmse, r2, n, path = result
        comparison_rows.append({"Evaluation": "Fixed random split", "Model": "MGT", "Test MAE": mae, "Test RMSE": rmse, "Test R²": r2, "n_test": n})

# Water hold-out: ALIGNN
if "alignn_water_metrics" in globals():
    row = alignn_water_metrics[alignn_water_metrics["Split"] == "test"].iloc[0]
    comparison_rows.append({
        "Evaluation": "Water hold-out",
        "Model": "ALIGNN",
        "Test MAE": row["MAE"],
        "Test RMSE": row["RMSE"],
        "Test R²": row["R²"],
        "n_test": int(row["n"]),
    })
else:
    result = _test_metrics_from_predictions(["predictions_alignn_waterholdout.csv", "predictions_alignn_waterholdout_.csv"])
    if result is not None:
        mae, rmse, r2, n, path = result
        comparison_rows.append({"Evaluation": "Water hold-out", "Model": "ALIGNN", "Test MAE": mae, "Test RMSE": rmse, "Test R²": r2, "n_test": n})

# Water hold-out: MGT
if "mgt_water_metrics" in globals():
    row = mgt_water_metrics[mgt_water_metrics["Split"] == "test"].iloc[0]
    comparison_rows.append({
        "Evaluation": "Water hold-out",
        "Model": "MGT",
        "Test MAE": row["MAE"],
        "Test RMSE": row["RMSE"],
        "Test R²": row["R²"],
        "n_test": int(row["n"]),
    })
else:
    result = _test_metrics_from_predictions(["predictions_mgt_waterholdout.csv", "predictions_mgt_waterholdout_.csv"])
    if result is not None:
        mae, rmse, r2, n, path = result
        comparison_rows.append({"Evaluation": "Water hold-out", "Model": "MGT", "Test MAE": mae, "Test RMSE": rmse, "Test R²": r2, "n_test": n})

final_model_comparison = pd.DataFrame(comparison_rows)
final_model_comparison = final_model_comparison.sort_values(["Evaluation", "Test MAE"]).reset_index(drop=True)

display(final_model_comparison.round(4))

best_by_setting = (
    final_model_comparison
    .sort_values("Test MAE")
    .groupby("Evaluation", as_index=False)
    .first()[["Evaluation", "Model", "Test MAE", "Test RMSE", "Test R²"]]
)

print("Best model by MAE in each evaluation setting:")
display(best_by_setting.round(4))

### Plot presentation note

The true-vs-predicted parity plots use matched x/y limits and an equal aspect ratio so that distance from the diagonal identity line is visually comparable across models. Residuals are defined as `dG_pred - dG_true`; positive residuals therefore indicate over-prediction and negative residuals indicate under-prediction.

### Final comparison summary

On the fixed random split, MGT is the stronger graph model, with lower test MAE and RMSE than ALIGNN. On the water hold-out extrapolation test, ALIGNN is slightly stronger by MAE, while both graph models show a much larger error than on the random split. This supports the main interpretation of Notebook 3: learned graph representations are highly competitive on random-split interpolation, but water remains the most difficult out-of-distribution solvent case.

**Why the two settings differ so much.** In the fixed random split, only the solute–solvent *pairs* are partitioned; the individual solute and solvent molecules themselves recur across train, validation and test. The encoder therefore sees almost every molecule during training, and a test pair is usually a new *combination* of already-seen molecules. This is why training error is very low (e.g. MGT train MAE ≈ 0.08 kcal mol$^{-1}$) and why the random-split test metrics are so strong: the task is pair *interpolation*, not generalisation to unseen molecules. The water hold-out is the first genuine extrapolation — the water-as-solvent relationship is absent from training entirely — which is exactly why both graph models, and the Notebook 2 descriptor baseline, degrade sharply on it. The two settings should therefore be read as complementary: random split measures interpolation quality, water hold-out measures out-of-distribution transfer.

## Appendix — Reproducibility and file locations

This notebook uses a **two-environment workflow**. Dataset preparation and notebook analysis were carried out in **JupyterHub**, while the full ALIGNN and MGT training jobs were run on **Apocrita/Andrena** using Slurm GPU jobs.

### JupyterHub (notebook side)

Working directory: the repository root (or another project directory)

Key files:

- `data/ML_Gibbs_Full_Database.csv` — source dataset (6,239 solute–solvent pairs, 949 unique molecules)
- `conformers.pkl` — cached 3D conformers for the 949 unique molecules
- `notebook3_config/structures.json` — pymatgen structures per molecule
- `notebook3_config/training_config.json` — shared hyperparameters and target scaling
- `build_canonical_splits.py` — canonical split builder (see below)

**Split definitions.** Every evaluation protocol is defined by split files shared by *both* the Notebook 2 classical pipeline and the Notebook 3 graph pipeline, so each protocol is a like-for-like comparison. Section 13 verifies this programmatically by asserting that the `row_idx` set of each graph prediction file matches the canonical split's test rows.

- `split_indices.npz` / `notebook3_config/pair_index.csv` — fixed random 80/10/10 split (4,991 / 624 / 624), stratified by `Solvent Type`, `RANDOM_STATE = 42`
- `notebook3_entity_splits/scaffold_disjoint/pair_index.csv` — 5,036 / 833 / 370
- `notebook3_entity_splits/leave_solvents_out/pair_index.csv` — 5,335 / 413 / 491
- `notebook3_entity_splits/leave_solutes_out/pair_index.csv` — 4,993 / 617 / 629

Each `pair_index.csv` carries a `dG_scaled` column standardised on **that protocol's training rows only**, using the sample standard deviation (`ddof = 1`). This was verified to floating-point precision (maximum deviation ≈ 1e-15) before the jobs were submitted, so no test-set statistics leak into the target scaling of any run.

> **Superseded artefact.** `build_canonical_splits.py` also produced a second folder, `notebook3_splits/`, together with `split_manifest.json`. Because those splits were generated independently, their entity-disjoint row counts differ (e.g. scaffold-disjoint 5,000 / 586 / 653) and they are **not** used for any reported result. They are retained only for provenance. Section 13 reads *only* the `fixed_random` entry of `split_manifest.json` (4,991 / 624 / 624, which does match) and ignores its entity-disjoint entries for this reason. All entity-disjoint results in this notebook and in Notebook 2 use `notebook3_entity_splits/`.

### Apocrita/Andrena (training side)

Project directory: configure a project-relative path for the HPC environment

- `train_alignn_pair.py`, `train_mgt_gpushort.py` — training scripts
- `train_mgt_disjoint.py` — patched MGT script exposing `DATA_DIR` and `TIME_BUDGET_SECONDS` as environment variables so each split trains in an isolated directory
- `submit_disjoint_graph_jobs.sh` — Slurm submission script for the six entity-disjoint jobs
- `graph_disjoint_runs/`, `mgt_disjoint_runs/` — per-split working directories
- `graph_disjoint_logs/` — Slurm logs, named `{model}_{split}.o{jobid}`

Prediction files (all split-suffixed, so no run overwrites another):

- `predictions_alignn.csv`, `predictions_mgt_full.csv` — fixed random-split predictions
- `predictions_alignn_waterholdout.csv`, `predictions_mgt_waterholdout.csv` — water hold-out predictions
- `graph_disjoint_results/predictions_{alignn,mgt}_{scaffold_disjoint,leave_solvents_out,leave_solutes_out}.csv` — entity-disjoint predictions
- `graph_disjoint_results/training_log_mgt_{split}.csv` — MGT per-epoch logs, used in Section 13 to test whether any MGT run was budget-limited

### Determinism and regeneration

- The fixed random split is stored in `split_indices.npz`; if unavailable it can be recreated with the `RANDOM_STATE = 42` logic used in Section 2. The entity-disjoint splits should be **reused from `notebook3_entity_splits/` rather than regenerated**: `build_canonical_splits.py` uses the same seed but a regenerated folder is not row-identical (see the note above), and regenerating would invalidate the comparison with the completed Notebook 2 disjoint runs.
- The conformer cache is stored in `conformers.pkl`. If unavailable, conformers can be regenerated (ETKDGv3 + MMFF94/UFF), although this takes longer and, because conformer embedding and GPU training are not fully deterministic, regenerated runs may differ slightly from the saved artefacts.
- The Slurm logs verify each GPU run: CUDA availability, graph construction for all 949 molecules, trainable-parameter counts (ALIGNN 4,174,593; MGT 2,279,777), the loaded split sizes, and the final metrics.

### Scope of the reported metrics

**Single seed.** The ALIGNN and MGT results are **single fixed-split, single-seed (42) training runs**, whereas the Notebook 2 classical baselines additionally reported a 10-fold cross-validation robustness check. The graph-model metrics should therefore be read as point estimates: small gaps — for example MGT (2.866) versus ALIGNN (2.713) kcal mol⁻¹ on the water hold-out, or ALIGNN (0.3157) versus the descriptor SVR (0.3228) on scaffold-disjoint — should not be over-interpreted as a definitive ranking without repeated seeds.

**Training duration is not a confound.** The entity-disjoint MGT jobs were given a 13-hour wall-clock budget (`TIME_BUDGET_SECONDS = 46800`), but the training logs show all three converged by **early stopping** well inside it (32, 47 and 38 epochs, with best validation MAE at epochs 21, 36 and 27 respectively). The Section 13 budget diagnostic re-checks this at run time. The MGT-versus-ALIGNN differences therefore reflect architecture and data, not compute.

**Water hold-out training sets differ in size.** The Notebook 2 descriptor water baseline trained on 5,597 rows (train + validation, no separate validation split), whereas the graph water hold-out runs trained on 5,037 rows with 560 held out for validation. The descriptor model therefore saw roughly 11% more training data on this protocol. The asymmetry is conservative *for* the graph models, but the 2.809 / 2.713 / 2.866 kcal mol⁻¹ comparison is not perfectly matched.

**Leave-solvents-out is a chemically narrow test set.** Because whole solvents are indivisible groups, its test set is approximately 45% hydrocarbons and 0.25% organic acids. The comparatively low MAE that every model family achieves on this protocol reflects that composition, and should not be read as evidence that holding out solvents is an easy task in general.

**Validation is a noisier proxy under group splits.** In the entity-disjoint protocols the validation and test sets hold out *different* entity groups, so validation MAE is not an unbiased estimate of test MAE. This is most visible for MGT on leave-solvents-out (best validation MAE 0.141 versus test MAE 0.270), and it means checkpoint selection is noisier here than in the fixed-random setting.


## Section 13 — Final audit: fixed random, water hold-out, and entity-disjoint graph comparisons

This final audit section loads the completed graph prediction CSVs from Apocrita and combines them with the best Notebook 2 descriptor baselines. It reports one headline table and one grouped MAE bar chart for the dissertation comparison.

The entity-disjoint graph results use the same canonical test rows as the Notebook 2 disjoint descriptor runs, so the descriptor / ALIGNN / MGT comparison is like-for-like for leave-solvents-out, leave-solutes-out, and scaffold-disjoint protocols.


In [ ]:
# Section 13 — Final audit table and dissertation-ready grouped MAE chart

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:  # plain-python fallback
    display = print



# File discovery
#
# Project-relative locations only. (An earlier draft also searched a
# hard-coded personal Downloads path; removed for portability.)

SEARCH_DIRS = [
    Path("."),
    Path("graph_disjoint_results"),
    Path("notebook3_config"),
]

CANONICAL_SPLIT_DIR = Path("notebook3_entity_splits")


def find_file(*names):
    """Return the first existing file among `names` across the search folders."""
    for folder in SEARCH_DIRS:
        for name in names:
            path = folder / name
            if path.exists():
                return path
    return None


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    r2 = 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)
    return mae, rmse, r2


# Canonical test-row identity check
#
# `expected_n` only checks the SIZE of the test set. Section 12 claims the
# graph runs used the *same canonical test rows* as the Notebook 2 descriptor
# runs, so we verify the actual row_idx sets against the canonical
# pair_index.csv. This is the check that substantiates the like-for-like
# comparison; a size check alone would pass a file with the right row count
# but the wrong rows.


PROTOCOL_TO_SPLIT_DIR = {
    "Scaffold-disjoint": "scaffold_disjoint",
    "Leave-solvents-out": "leave_solvents_out",
    "Leave-solutes-out": "leave_solutes_out",
}


def canonical_test_row_idx(protocol):
    """Test-set row_idx values from the canonical split, or None if unavailable."""
    split_dir = PROTOCOL_TO_SPLIT_DIR.get(protocol)
    if split_dir is None:
        return None
    path = CANONICAL_SPLIT_DIR / split_dir / "pair_index.csv"
    if not path.exists():
        return None
    canon = pd.read_csv(path)
    return set(canon.loc[canon["split"] == "test", "row_idx"].astype(int))


def graph_test_row(protocol, model, filename_candidates, expected_n=None):
    path = find_file(*filename_candidates)
    if path is None:
        raise FileNotFoundError(
            f"Missing prediction file for {model} / {protocol}: {filename_candidates}"
        )

    preds = pd.read_csv(path)
    required = {"row_idx", "split", "dG_true", "dG_pred"}
    missing = required - set(preds.columns)
    assert not missing, f"{path.name} missing columns: {missing}"

    test = preds[preds["split"] == "test"].copy()

    # (a) size check
    if expected_n is not None:
        assert len(test) == expected_n, (
            f"{path.name}: expected {expected_n} test rows, found {len(test)}"
        )

    # (b) identity check against the canonical split
    expected_idx = canonical_test_row_idx(protocol)
    if expected_idx is not None:
        got_idx = set(test["row_idx"].astype(int))
        assert got_idx == expected_idx, (
            f"{path.name}: test row_idx set does not match the canonical "
            f"{PROTOCOL_TO_SPLIT_DIR[protocol]} split "
            f"({len(got_idx - expected_idx)} unexpected, "
            f"{len(expected_idx - got_idx)} missing rows)"
        )
        row_check = "row_idx verified"
    else:
        row_check = "row_idx not verified (canonical file unavailable)"

    mae, rmse, r2 = regression_metrics(test["dG_true"], test["dG_pred"])
    return {
        "Protocol": protocol,
        "Model family": model,
        "Model": model,
        "Representation": "Learned molecular graphs",
        "n_test": len(test),
        "Test MAE": mae,
        "Test RMSE": rmse,
        "Test R²": r2,
        "Source file": path.name,
        "Row check": row_check,
    }


def descriptor_row_from_summary(protocol, split_name, fallback):
    """
    Load the best Notebook 2 descriptor result for an entity-disjoint split.

    The run_summary_*.json files were written primarily to record split sizes,
    so their keys may not include the best-model metrics. We therefore fall back
    to the values from the completed Notebook 2 run BOTH when the file is absent
    AND when it is present but missing the expected keys — a bare KeyError would
    otherwise crash the whole audit cell.
    """
    candidates = [
        f"run_summary_{split_name}.json",
        f"run_summary_{split_name} (1).json",
    ]
    path = find_file(*candidates)

    data, source = None, None
    if path is not None:
        try:
            with open(path) as f:
                summary = json.load(f)
            data = {
                "Model": summary["best_model"],
                "Representation": summary["best_features"],
                "n_test": int(summary["n_test"]),
                "Test MAE": float(summary["best_test_mae"]),
                "Test RMSE": float(summary["best_test_rmse"]),
                "Test R²": float(summary["best_test_r2"]),
            }
            source = path.name
        except (KeyError, ValueError, json.JSONDecodeError) as exc:
            warnings.warn(
                f"{path.name} present but unusable ({type(exc).__name__}: {exc}); "
                f"using recorded Notebook 2 values for {protocol}."
            )
            data = None

    if data is None:
        data = dict(fallback)
        source = "recorded value from completed Notebook 2 run"

    return {
        "Protocol": protocol,
        "Model family": "Descriptor",
        "Model": data["Model"],
        "Representation": data["Representation"],
        "n_test": data["n_test"],
        "Test MAE": data["Test MAE"],
        "Test RMSE": data["Test RMSE"],
        "Test R²": data["Test R²"],
        "Source file": source,
        "Row check": "canonical split (shared with graph runs)",
    }



# Best Notebook 2 descriptor baselines


descriptor_rows = [
    {
        "Protocol": "Fixed random",
        "Model family": "Descriptor",
        "Model": "SVR-rbf",
        "Representation": "RDKit descriptors",
        "n_test": 624,
        "Test MAE": 0.2113,
        "Test RMSE": 0.4967,
        "Test R²": 0.9513,
        "Source file": "Notebook 2 fixed random summary",
        "Row check": "canonical split (shared with graph runs)",
    },
    {
        "Protocol": "Water hold-out",
        "Model family": "Descriptor",
        "Model": "RandomForest",
        "Representation": "RDKit descriptors",
        "n_test": 642,
        "Test MAE": 2.8091,
        "Test RMSE": 3.7855,
        "Test R²": 0.0306,
        "Source file": "Notebook 2 water hold-out summary",
        "Row check": "same water test rows; see training-set caveat in appendix",
    },
    descriptor_row_from_summary(
        "Leave-solutes-out",
        "leave_solutes_out",
        {
            "Model": "SVR-rbf",
            "Representation": "Descriptors",
            "n_test": 629,
            "Test MAE": 0.29138250442149854,
            "Test RMSE": 0.4815214639375437,
            "Test R²": 0.9179370931366309,
        },
    ),
    descriptor_row_from_summary(
        "Leave-solvents-out",
        "leave_solvents_out",
        {
            "Model": "RandomForest",
            "Representation": "Descriptors",
            "n_test": 491,
            "Test MAE": 0.2534279149421983,
            "Test RMSE": 0.35641616729580616,
            "Test R²": 0.9616566373760781,
        },
    ),
    descriptor_row_from_summary(
        "Scaffold-disjoint",
        "scaffold_disjoint",
        {
            "Model": "SVR-rbf",
            "Representation": "Descriptors",
            "n_test": 370,
            "Test MAE": 0.3228336181426491,
            "Test RMSE": 0.761210305544254,
            "Test R²": 0.8390401107051927,
        },
    ),
]



# Graph model rows loaded directly from prediction CSVs


graph_rows = [
    graph_test_row("Fixed random", "ALIGNN",
                   ["predictions_alignn.csv", "predictions_alignn_.csv"],
                   expected_n=624),
    graph_test_row("Fixed random", "MGT",
                   ["predictions_mgt_full.csv", "predictions_mgt_full_.csv"],
                   expected_n=624),
    graph_test_row("Water hold-out", "ALIGNN",
                   ["predictions_alignn_waterholdout.csv",
                    "predictions_alignn_waterholdout_.csv"],
                   expected_n=642),
    graph_test_row("Water hold-out", "MGT",
                   ["predictions_mgt_waterholdout.csv",
                    "predictions_mgt_waterholdout_.csv"],
                   expected_n=642),
    graph_test_row("Leave-solutes-out", "ALIGNN",
                   ["predictions_alignn_leave_solutes_out.csv"], expected_n=629),
    graph_test_row("Leave-solutes-out", "MGT",
                   ["predictions_mgt_leave_solutes_out.csv"], expected_n=629),
    graph_test_row("Leave-solvents-out", "ALIGNN",
                   ["predictions_alignn_leave_solvents_out.csv"], expected_n=491),
    graph_test_row("Leave-solvents-out", "MGT",
                   ["predictions_mgt_leave_solvents_out.csv"], expected_n=491),
    graph_test_row("Scaffold-disjoint", "ALIGNN",
                   ["predictions_alignn_scaffold_disjoint.csv"], expected_n=370),
    graph_test_row("Scaffold-disjoint", "MGT",
                   ["predictions_mgt_scaffold_disjoint.csv"], expected_n=370),
]


protocol_order = [
    "Fixed random",
    "Leave-solvents-out",
    "Leave-solutes-out",
    "Scaffold-disjoint",
    "Water hold-out",
]
model_order = ["Descriptor", "ALIGNN", "MGT"]

final_audit_table = pd.DataFrame(descriptor_rows + graph_rows)
final_audit_table["Protocol"] = pd.Categorical(
    final_audit_table["Protocol"], categories=protocol_order, ordered=True)
final_audit_table["Model family"] = pd.Categorical(
    final_audit_table["Model family"], categories=model_order, ordered=True)
final_audit_table = (
    final_audit_table.sort_values(["Protocol", "Model family"]).reset_index(drop=True)
)

display_cols = [
    "Protocol", "Model family", "Model", "n_test",
    "Test MAE", "Test RMSE", "Test R²", "Source file",
]

print("Final audit headline table")
display(final_audit_table[display_cols].round(4))

print("\nCanonical test-row verification")
display(final_audit_table[["Protocol", "Model family", "Row check"]])



# Dataset duplicate check
#
# `straddling_pairs == 0` shows that no duplicate pair crosses a split
# boundary, but a duplicate sitting entirely inside `train` would never
# straddle. Checking the source dataset directly settles the question.


dataset_path = find_file("data/ML_Gibbs_Full_Database.csv", "ML_Gibbs_Full_Database (1).csv")
if dataset_path is not None:
    raw_df = pd.read_csv(dataset_path)
    duplicate_pair_rows = int(
        raw_df.duplicated(subset=["Solute SMILES", "Solvent SMILES"]).sum()
    )
    duplicate_full_rows = int(raw_df.duplicated().sum())
    print("\nDuplicate-pair check")
    print(f"Loaded dataset: {dataset_path.name}")
    print(f"Duplicate solute-solvent pair rows: {duplicate_pair_rows}")
    print(f"Fully duplicated rows: {duplicate_full_rows}")
    if duplicate_pair_rows == 0:
        print(
            "The dataset contains no duplicate (solute, solvent) pairs, so the "
            "multi-source `Database Origin` merge introduces no row-level leakage "
            "into any split."
        )
    else:
        print(
            "Duplicate pairs exist. Because every protocol reported zero straddling "
            "pairs, no duplicate crosses a split boundary, but the honest phrasing is "
            "'no duplicate pair straddles a split' rather than 'no duplicates exist'."
        )
else:
    print(
        "\nDataset duplicate-pair check skipped: data/ML_Gibbs_Full_Database.csv was not "
        "found in the configured repository search directories."
    )



# Entity-overlap evidence for the interpolation-vs-extrapolation claim
#
# Read from the fixed_random entry of split_manifest.json when available.
# NOTE: only the fixed_random entry is used. The manifest's entity-disjoint
# entries describe the SUPERSEDED `notebook3_splits/` folder (e.g. scaffold
# 5000/586/653), not the `notebook3_entity_splits/` splits used here, so
# reading them would silently contradict the results table above.


FIXED_RANDOM_OVERLAP = {"solute": 0.1502, "solvent": 0.0187}  # recorded values

manifest_path = find_file("split_manifest.json")
if manifest_path is not None:
    try:
        with open(manifest_path) as f:
            fr = json.load(f)["protocols"]["fixed_random"]
        if (int(fr["n_train"]), int(fr["n_val"]), int(fr["n_test"])) == (4991, 624, 624):
            FIXED_RANDOM_OVERLAP = {
                "solute": fr["solute_overlap"]["unseen_fraction"],
                "solvent": fr["solvent_overlap"]["unseen_fraction"],
            }
            print(f"\nEntity-overlap figures read from {manifest_path.name} "
                  f"(fixed_random entry only).")
        else:
            print("\nsplit_manifest.json fixed_random sizes do not match "
                  "(4991/624/624); using recorded overlap values instead.")
    except (KeyError, ValueError, json.JSONDecodeError):
        print("\nsplit_manifest.json unusable; using recorded overlap values.")
else:
    print("\nsplit_manifest.json not found; using recorded overlap values.")

pct_solute_seen = 100 * (1 - FIXED_RANDOM_OVERLAP["solute"])
pct_solvent_seen = 100 * (1 - FIXED_RANDOM_OVERLAP["solvent"])
print(
    f"Fixed random split: {pct_solute_seen:.1f}% of test solutes and "
    f"{pct_solvent_seen:.1f}% of test solvents were already seen in training "
    f"(vs 0% by construction for the held-out entity in each disjoint split)."
)



# MGT budget / early-stopping diagnostic
#
# The per-epoch validation column has been written as `val_mae` and `val_MAE`
# in different runs, so match case-insensitively rather than asserting a name.


def _find_val_mae_col(log):
    for c in log.columns:
        lc = c.lower()
        if "val" in lc and "mae" in lc:
            return c
    return None


def _find_epoch_col(log):
    for c in log.columns:
        if c.lower().startswith("epoch"):
            return c
    return None


def mgt_budget_row(protocol, split_name):
    path = find_file(f"training_log_mgt_{split_name}.csv")
    if path is None:
        return None

    log = pd.read_csv(path)
    val_col = _find_val_mae_col(log)
    epoch_col = _find_epoch_col(log)
    if val_col is None:
        warnings.warn(f"{path.name}: no validation-MAE column found; skipping.")
        return None
    if epoch_col is None:
        log = log.reset_index().rename(columns={"index": "epoch"})
        epoch_col = "epoch"

    best_idx = log[val_col].idxmin()
    best_epoch = int(log.loc[best_idx, epoch_col])
    last_epoch = int(log[epoch_col].max())
    budget_limited = best_epoch >= last_epoch - 4

    return {
        "Protocol": protocol,
        "epochs run": len(log),
        "best epoch": best_epoch,
        "last epoch": last_epoch,
        "best val MAE": float(log.loc[best_idx, val_col]),
        "final val MAE": float(log.iloc[-1][val_col]),
        "budget verdict": "YES — budget-limited" if budget_limited else "no — converged",
        "Source file": path.name,
    }


mgt_budget_rows = [
    mgt_budget_row("Scaffold-disjoint", "scaffold_disjoint"),
    mgt_budget_row("Leave-solvents-out", "leave_solvents_out"),
    mgt_budget_row("Leave-solutes-out", "leave_solutes_out"),
]
mgt_budget_rows = [row for row in mgt_budget_rows if row is not None]

if mgt_budget_rows:
    mgt_budget_diagnostic = pd.DataFrame(mgt_budget_rows)
    print("\nMGT budget diagnostic")
    display(mgt_budget_diagnostic.round(4))

    if (mgt_budget_diagnostic["budget verdict"] == "YES — budget-limited").any():
        print(
            "At least one MGT run appears budget-limited. Interpret that protocol's "
            "MGT test metric as a lower bound on what the configuration might achieve "
            "with longer training."
        )
    else:
        print(
            "All three entity-disjoint MGT runs converged by early stopping before the "
            "wall-clock cap. The MGT-vs-ALIGNN gaps should therefore not be explained "
            "as a simple compute-budget artefact."
        )

    mgt_test_only = final_audit_table[
        (final_audit_table["Model family"] == "MGT")
        & (final_audit_table["Protocol"].isin(mgt_budget_diagnostic["Protocol"]))
    ][["Protocol", "Test MAE"]]
    mgt_val_test = mgt_budget_diagnostic.merge(mgt_test_only, on="Protocol", how="left")
    mgt_val_test["test / best-val MAE"] = (
        mgt_val_test["Test MAE"] / mgt_val_test["best val MAE"]
    )
    print("\nMGT validation-to-test transfer diagnostic")
    display(
        mgt_val_test[["Protocol", "best val MAE", "Test MAE", "test / best-val MAE"]].round(4)
    )
else:
    print("\nMGT budget diagnostic skipped: no training_log_mgt_*.csv files found.")



# Save table and plot


output_dir = Path("notebook3_final_audit_outputs")
output_dir.mkdir(exist_ok=True)
table_path = output_dir / "notebook3_final_audit_headline_table.csv"
plot_path = output_dir / "notebook3_final_audit_mae_bar_chart.png"

final_audit_table.to_csv(table_path, index=False)

mae_plot = (
    final_audit_table
    .pivot(index="Protocol", columns="Model family", values="Test MAE")
    .loc[protocol_order, model_order]
)

plt.figure(figsize=(11, 5.5))
ax = mae_plot.plot(
    kind="bar",
    ax=plt.gca(),
    color={"Descriptor": "#6c757d", "ALIGNN": "#1f77b4", "MGT": "#2ca02c"},
    width=0.78,
)
ax.set_ylabel("Test MAE (kcal/mol)")
ax.set_xlabel("")
ax.set_title("Descriptor vs graph models across evaluation protocols")
ax.grid(axis="y", alpha=0.25)
ax.legend(title="")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved table: {table_path}")
print(f"Saved figure: {plot_path}")



# Compact interpretation for dissertation notes

best_by_protocol = (
    final_audit_table
    .sort_values("Test MAE")
    .groupby("Protocol", observed=True)
    .first()[["Model family", "Model", "Test MAE", "Test RMSE", "Test R²"]]
    .reset_index()
)
print("\nBest model by protocol (all families):")
display(best_by_protocol.round(4))

best_graph = (
    final_audit_table[final_audit_table["Model family"] != "Descriptor"]
    .sort_values("Test MAE")
    .groupby("Protocol", observed=True)
    .first()[["Model family", "Test MAE"]]
    .reset_index()
)
print("\nBest graph model by protocol:")
display(best_graph.round(4))

disjoint = ["Leave-solvents-out", "Leave-solutes-out", "Scaffold-disjoint"]
descriptor_wins = best_by_protocol[
    best_by_protocol["Protocol"].isin(disjoint)
    & (best_by_protocol["Model family"] == "Descriptor")
]

print(
    "\nInterpretation:\n"
    f"1. The fixed random split remains the most optimistic setting: "
    f"{pct_solute_seen:.1f}% of test solutes and {pct_solvent_seen:.1f}% of test "
    "solvents were already seen during training, so a test pair is usually a new "
    "COMBINATION of known molecules. It measures pair interpolation more than true "
    "entity extrapolation.\n"
    f"2. The best Notebook 2 descriptor baseline is strongest on "
    f"{len(descriptor_wins)} of the {len(disjoint)} entity-disjoint protocols "
    f"({', '.join(descriptor_wins['Protocol'].astype(str)) or 'none'}). This should "
    "not be overstated: ALIGNN is the strongest model on scaffold-disjoint, and is "
    "especially better there by RMSE and R². The fair claim is that learned graph "
    "representations are competitive with, but do not clearly surpass, well-tuned "
    "RDKit pair descriptors under entity-disjoint evaluation on a dataset of this "
    "size (6,239 pairs / 949 molecules).\n"
    "3. Between the graph models, MGT is strongest for leave-solutes-out, whereas "
    "ALIGNN is strongest for leave-solvents-out and scaffold-disjoint. The MGT "
    "training logs indicate early-stopping convergence rather than a wall-clock "
    "budget cutoff, so the scaffold-disjoint MGT-vs-ALIGNN gap is not a simple "
    "compute-time artefact. Because the pair model uses late fusion with no "
    "cross-molecule Coulomb edges, MGT's long-range attention pathway carries no "
    "solute-solvent interaction information, so no advantage over ALIGNN is expected "
    "on this task.\n"
    "4. The leave-solvents-out MGT validation MAE is much lower than its test MAE, "
    "which is expected under group splits because validation and test hold out "
    "different solvent groups. Validation is therefore a noisier proxy for test "
    "performance than in the fixed-random setting.\n"
    "5. Water remains the hardest setting for every model family (MAE 2.71-2.87 "
    "kcal/mol), confirming that aqueous solvation is an out-of-distribution regime "
    "rather than merely an unseen solvent label.\n"
    "\nCaveats (appendix): single seed (42); the Notebook 2 water baseline trained on "
    "train+val (5,597 rows) whereas the graph water runs trained on train only "
    "(5,037 rows); and the leave-solvents-out test set is chemically narrow "
    "(~45% hydrocarbons), which is why every model scores well on it."
)


### Final audit interpretation

**Evaluation protocol, not architecture, controls apparent performance.** The fixed random split is the most optimistic setting because only solute–solvent *pairs* are partitioned: roughly 85% of test solutes and 98% of test solvents were already seen during training, so a test pair is usually a new *combination* of known molecules. That protocol measures pair **interpolation**. The entity-disjoint protocols hold out 100% of the relevant entity by construction, and the water hold-out removes an entire solvation regime; those measure **extrapolation**.

**Graph representations are competitive with, but do not clearly surpass, classical descriptors under entity-disjoint evaluation.** Both graph models beat the descriptor baseline on the fixed random split (MGT 0.1532, ALIGNN 0.2050 vs SVR-rbf 0.2113 kcal mol⁻¹). Once entities are held out, the best Notebook 2 descriptor model is strongest on two of the three entity-disjoint protocols — leave-solvents-out (0.2534 vs ALIGNN 0.2611, MGT 0.2700) and leave-solutes-out (0.2914 vs MGT 0.3749, ALIGNN 0.4279). This should not be overstated in the other direction, however: **ALIGNN is the strongest model on scaffold-disjoint**, marginally by MAE (0.3157 vs 0.3228) but clearly by RMSE (0.5219 vs 0.7612) and R² (0.9243 vs 0.8390), indicating that the descriptor SVR has a heavy error tail on unseen scaffolds that ALIGNN does not. The defensible dissertation claim is therefore that learned graph representations are competitive, that their apparent advantage on the random split is largely an artefact of entity overlap, and that on a dataset of this size (6,239 pairs over 949 molecules) well-tuned RDKit pair descriptors remain a strong baseline.

**Between the two graph models there is no consistent winner, and MGT's mechanism cannot help on this task.** MGT is stronger on leave-solutes-out; ALIGNN on leave-solvents-out and scaffold-disjoint. The MGT training logs show that all three entity-disjoint runs converged by early stopping (32, 47 and 38 epochs) well before the 13-hour wall-clock cap, so the scaffold-disjoint gap is **not** a compute-budget artefact. The more likely explanation is architectural: because the pair model uses two-encoder late fusion with **no cross-molecule edges**, MGT's global/Coulomb graph operates only *within* each isolated molecule. Its long-range electrostatic pathway therefore supplies no solute–solvent interaction information and adds capacity without adding signal. Testing the MGT thesis properly on solvation would require global edges spanning the solute and the solvent, which is the natural next experiment.

**Water remains the hardest setting for every model family.** All three models degrade to MAE ≈ 2.71–2.87 kcal mol⁻¹, confirming that aqueous solvation is a genuinely out-of-distribution regime rather than merely an unseen solvent label. ALIGNN (2.713) and MGT (2.866) bracket the descriptor baseline (2.809), and MGT attains a better RMSE (3.419 vs 3.786) and R² (0.209 vs 0.031) despite its slightly worse MAE — so no model family transfers to water.

**On data leakage.** Section 13 checks the source dataset directly for duplicate (solute, solvent) pairs, and every protocol reported zero pairs straddling a split boundary. Target scaling was fitted on training rows only, per protocol, with `ddof = 1`. The strong fixed-random metrics are therefore the result of entity overlap — an inherent property of splitting pairs rather than molecules — and not of improper data handling.

**Caveats** (detailed in the appendix): all graph results are single-seed (42); the Notebook 2 water baseline trained on ~11% more data (train + validation, 5,597 rows) than the graph water runs (5,037 rows); the leave-solvents-out test set is ~45% hydrocarbons, which is why every model scores well on it; and under group splits validation is a noisier proxy for test performance, most visibly for MGT on leave-solvents-out (validation MAE 0.141 versus test MAE 0.270).
